# Notebook 1

## Scientific Objective
Evaluate whether resistance prediction arises from **true spatial pathway communication** rather than shortcut correlations.

## Hypothesis
If the model is biologically meaningful:
- Performance should degrade under falsification (spatial/pathway disruption)
- Performance should recover under correct structured signaling

## Structure
1. Data / simulation design
2. Model construction
3. Falsification experiments
4. Quantitative evaluation
5. Interpretation

---


### Step 0

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [1]:
# CELL 1 — setup, imports, deterministic behavior, and falsification config

from pathlib import Path
import os
import json
import math
import copy
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

import scanpy as sc
import anndata as ad

from scipy import sparse
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, pearsonr

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.utils import degree

warnings.filterwarnings("ignore")

# ---------------------------
# Reproducibility
# ---------------------------
def set_seed(seed: int = 17) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(17)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# Paths
# ---------------------------
PROJECT_ROOT = Path.cwd()
OUTDIR = PROJECT_ROOT / "prototype_falsification_outputs"
TABLES = OUTDIR / "tables"
PLOTS = OUTDIR / "plots"
ARTIFACTS = OUTDIR / "artifacts"
LOGS = OUTDIR / "logs"

for d in [OUTDIR, TABLES, PLOTS, ARTIFACTS, LOGS]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------
# Falsification design config
# ---------------------------
CONFIG = {
    "run_name": "prototype_falsification_v1",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "device": str(DEVICE),

    # worlds for kill-test
    "worlds": ["true", "mask_shuffle", "spatial_null"],

    # seeds for stability check
    "seeds": [11, 17, 23],

    # prototype biology
    "seeded_axes": [
        ("CXCL12", "CXCR4"),
        ("TGFB1", "TGFBR2"),
        ("IFNG", "IFNGR1"),   # negative-control-like axis in the original prototype
    ],
    "expected_pathway_direction": {
        "TGFB": "up",
        "WNT": "up",
        "IFNG": "down",
        "Antigen_Presentation": "down",
    },

    # graph / training settings matching the prototype design
    "graph_k": 8,
    "epochs": 50,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "hidden_dim": 64,
    "gat_heads": 4,
    "dropout": 0.2,

    # falsification controls
    "top_attention_quantile": 0.95,
    "top_rank_threshold": 10,
    "num_random_decoys": 25,
    "num_expr_matched_decoys": 25,
    "preserve_degree_in_spatial_null": True,
    "preserve_pathway_sizes_in_mask_shuffle": True,

    # saving
    "save_tables": True,
    "save_plots": True,
}

with open(ARTIFACTS / "run_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Setup complete.")
print(f"Device: {DEVICE}")
print(f"Output directory: {OUTDIR.resolve()}")
print(json.dumps(CONFIG, indent=2))

Setup complete.
Device: cpu
Output directory: /Users/sally/Desktop/prototype_falsification_outputs
{
  "run_name": "prototype_falsification_v1",
  "created_at": "2026-03-12T19:40:38",
  "device": "cpu",
  "worlds": [
    "true",
    "mask_shuffle",
    "spatial_null"
  ],
  "seeds": [
    11,
    17,
    23
  ],
  "seeded_axes": [
    [
      "CXCL12",
      "CXCR4"
    ],
    [
      "TGFB1",
      "TGFBR2"
    ],
    [
      "IFNG",
      "IFNGR1"
    ]
  ],
  "expected_pathway_direction": {
    "TGFB": "up",
    "WNT": "up",
    "IFNG": "down",
    "Antigen_Presentation": "down"
  },
  "graph_k": 8,
  "epochs": 50,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "hidden_dim": 64,
  "gat_heads": 4,
  "dropout": 0.2,
  "top_attention_quantile": 0.95,
  "top_rank_threshold": 10,
  "num_random_decoys": 25,
  "num_expr_matched_decoys": 25,
  "preserve_degree_in_spatial_null": true,
  "preserve_pathway_sizes_in_mask_shuffle": true,
  "save_tables": true,
  "save_plots": true
}


#### Output interpretation
- Compare results to falsification expectations
- Look for separation between TRUE vs NULL conditions
- Strong separation indicates meaningful signal usage


### Step 1

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [3]:
# CELL 2 — create a synthetic spatial prototype dataset in-memory
# This replaces loading a real prototype dataset.
# Goal:
# - 3 spatial regions: tumor core, interface, stroma
# - seeded axes:
#     CXCL12 -> CXCR4   (interface/stroma chemotactic motif)
#     TGFB1  -> TGFBR2  (tumor/interface remodeling motif)
#     IFNG   -> IFNGR1  (weak negative-control-like axis)
# - pathway expectations:
#     TGFB, WNT up
#     IFNG, Antigen_Presentation down
#
# Output:
# - adata with counts in .X
# - adata.obsm["spatial"]
# - adata.obs region labels and designed latent state labels

# ---------------------------
# Synthetic prototype design
# ---------------------------
N_TUMOR = 900
N_INTERFACE = 300
N_STROMA = 700
N_TOTAL = N_TUMOR + N_INTERFACE + N_STROMA

# Core genes we care about
seed_genes = [
    "CXCL12", "CXCR4",
    "TGFB1", "TGFBR2",
    "IFNG", "IFNGR1",
]

pathway_genes = {
    "TGFB": ["SMAD2", "SMAD3", "SMAD4", "SERPINE1", "COL1A1", "COL1A2"],
    "WNT": ["CTNNB1", "FZD7", "WNT5A", "AXIN2", "MYC", "CCND1"],
    "IFNG": ["STAT1", "IRF1", "JAK1", "JAK2", "GBP1", "CXCL9"],
    "Antigen_Presentation": ["HLA-A", "HLA-B", "B2M", "TAP1", "PSMB8", "NLRC5"],
}

marker_genes = [
    "EPCAM", "KRT8", "KRT18",         # tumor-ish
    "COL3A1", "DCN", "LUM",           # stromal-ish
    "PECAM1", "VWF",                  # endothelial-ish
    "PTPRC", "CD3D", "NKG7",          # immune-ish
]

background_genes = [f"BG_GENE_{i:03d}" for i in range(1, 121)]

all_genes = seed_genes + sum(pathway_genes.values(), []) + marker_genes + background_genes

# preserve order, remove duplicates
seen = set()
gene_names = []
for g in all_genes:
    if g not in seen:
        gene_names.append(g)
        seen.add(g)

G = len(gene_names)
gene_to_idx = {g: i for i, g in enumerate(gene_names)}

# ---------------------------
# Spatial layout
# ---------------------------
# Tumor core: centered circle
# Interface: annulus
# Stroma: outer annulus
rng = np.random.default_rng(17)

def sample_disc(n, r_min, r_max, center=(0.0, 0.0)):
    theta = rng.uniform(0, 2*np.pi, size=n)
    # uniform in area
    rr = np.sqrt(rng.uniform(r_min**2, r_max**2, size=n))
    x = center[0] + rr * np.cos(theta)
    y = center[1] + rr * np.sin(theta)
    return np.column_stack([x, y])

coords_tumor = sample_disc(N_TUMOR, 0.0, 25.0)
coords_interface = sample_disc(N_INTERFACE, 25.0, 33.0)
coords_stroma = sample_disc(N_STROMA, 33.0, 60.0)

coords = np.vstack([coords_tumor, coords_interface, coords_stroma])

region = (
    ["tumor"] * N_TUMOR +
    ["interface"] * N_INTERFACE +
    ["stroma"] * N_STROMA
)

# Resistant-like latent state:
# stronger in interface and part of tumor, weak in stroma
resistant_score = np.zeros(N_TOTAL, dtype=float)
resistant_score[:N_TUMOR] = rng.beta(2.2, 2.0, size=N_TUMOR)
resistant_score[N_TUMOR:N_TUMOR+N_INTERFACE] = rng.beta(4.0, 1.5, size=N_INTERFACE)
resistant_score[N_TUMOR+N_INTERFACE:] = rng.beta(1.5, 3.5, size=N_STROMA)

# Binary version for easy interpretation
resistant_like = (resistant_score >= np.quantile(resistant_score, 0.6)).astype(int)

# ---------------------------
# Mean-expression design
# ---------------------------
# We generate Poisson counts from region/state-specific rates.
# This is a prototype, so "biology" is intentionally designed.

base_mu = np.full((N_TOTAL, G), 0.08, dtype=float)

# Region masks
tumor_mask = np.array(region) == "tumor"
interface_mask = np.array(region) == "interface"
stroma_mask = np.array(region) == "stroma"
res_mask = resistant_like == 1
nonres_mask = resistant_like == 0

def add_mu(gene, mask, value):
    base_mu[mask, gene_to_idx[gene]] += value

# Seeded LR biology
# CXCL12 high in stroma/interface; CXCR4 higher at interface and resistant-like
add_mu("CXCL12", stroma_mask, 2.8)
add_mu("CXCL12", interface_mask, 1.8)

add_mu("CXCR4", interface_mask, 2.0)
add_mu("CXCR4", tumor_mask & res_mask, 1.3)
add_mu("CXCR4", stroma_mask, 0.3)

# TGFB remodeling motif: stronger in tumor/interface, especially resistant-like
add_mu("TGFB1", tumor_mask, 1.2)
add_mu("TGFB1", interface_mask, 2.2)
add_mu("TGFB1", res_mask, 1.4)

add_mu("TGFBR2", interface_mask, 1.6)
add_mu("TGFBR2", tumor_mask & res_mask, 1.5)
add_mu("TGFBR2", stroma_mask, 0.4)

# IFNG weak / diffuse negative-control-like
add_mu("IFNG", tumor_mask, 0.15)
add_mu("IFNG", interface_mask, 0.12)
add_mu("IFNG", stroma_mask, 0.10)

add_mu("IFNGR1", tumor_mask, 0.25)
add_mu("IFNGR1", interface_mask, 0.20)
add_mu("IFNGR1", stroma_mask, 0.20)

# Pathway expectations
# TGFB, WNT up in resistant-like regions
for g in pathway_genes["TGFB"]:
    add_mu(g, res_mask, 1.0)
    add_mu(g, interface_mask, 0.5)

for g in pathway_genes["WNT"]:
    add_mu(g, res_mask, 0.9)
    add_mu(g, tumor_mask, 0.4)

# IFNG, Antigen Presentation down in resistant-like regions
for g in pathway_genes["IFNG"]:
    add_mu(g, nonres_mask, 0.9)
    add_mu(g, stroma_mask, 0.2)

for g in pathway_genes["Antigen_Presentation"]:
    add_mu(g, nonres_mask, 0.8)
    add_mu(g, tumor_mask & nonres_mask, 0.3)

# Marker structure
for g in ["EPCAM", "KRT8", "KRT18"]:
    add_mu(g, tumor_mask, 2.5)
    add_mu(g, interface_mask, 0.7)

for g in ["COL3A1", "DCN", "LUM"]:
    add_mu(g, stroma_mask, 2.7)
    add_mu(g, interface_mask, 1.2)

for g in ["PECAM1", "VWF"]:
    add_mu(g, interface_mask, 1.0)
    add_mu(g, stroma_mask, 0.8)

for g in ["PTPRC", "CD3D", "NKG7"]:
    add_mu(g, stroma_mask, 0.6)
    add_mu(g, interface_mask, 0.4)
    add_mu(g, tumor_mask, 0.1)

# Background genes: low random heterogeneity
for g in background_genes:
    base_mu[:, gene_to_idx[g]] += rng.uniform(0.0, 0.15, size=N_TOTAL)

# Library size heterogeneity
lib_factor = rng.lognormal(mean=0.0, sigma=0.35, size=N_TOTAL)
mu = base_mu * lib_factor[:, None]

# Sample counts
X = rng.poisson(mu).astype(np.float32)

# Make sparse for AnnData
X_sparse = sparse.csr_matrix(X)

# Build AnnData
adata = ad.AnnData(X=X_sparse)
adata.var_names = gene_names
adata.obs_names = [f"spot_{i:04d}" for i in range(N_TOTAL)]
adata.obsm["spatial"] = coords.astype(np.float32)

adata.obs["region"] = pd.Categorical(region, categories=["tumor", "interface", "stroma"])
adata.obs["resistant_score"] = resistant_score
adata.obs["resistant_like"] = pd.Categorical(
    np.where(resistant_like == 1, "resistant_like", "other")
)
adata.obs["in_tissue"] = 1

# Keep raw counts before any normalization
adata.layers["counts"] = adata.X.copy()

# ---------------------------
# QC summary for the synthetic dataset
# ---------------------------
region_counts = adata.obs["region"].value_counts().rename_axis("region").reset_index(name="n_spots")
expr_summary = pd.DataFrame({
    "gene": ["CXCL12", "CXCR4", "TGFB1", "TGFBR2", "IFNG", "IFNGR1"],
    "mean_count": [float(np.asarray(adata[:, g].X.mean()).squeeze()) for g in ["CXCL12", "CXCR4", "TGFB1", "TGFBR2", "IFNG", "IFNGR1"]],
})

dataset_summary = pd.DataFrame([{
    "n_spots": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "n_tumor": int((adata.obs["region"] == "tumor").sum()),
    "n_interface": int((adata.obs["region"] == "interface").sum()),
    "n_stroma": int((adata.obs["region"] == "stroma").sum()),
    "x_min": float(adata.obsm["spatial"][:, 0].min()),
    "x_max": float(adata.obsm["spatial"][:, 0].max()),
    "y_min": float(adata.obsm["spatial"][:, 1].min()),
    "y_max": float(adata.obsm["spatial"][:, 1].max()),
}])

if CONFIG["save_tables"]:
    dataset_summary.to_csv(TABLES / "synthetic_dataset_summary.csv", index=False)
    region_counts.to_csv(TABLES / "synthetic_region_counts.csv", index=False)
    expr_summary.to_csv(TABLES / "synthetic_seed_gene_expression.csv", index=False)

print("Synthetic prototype dataset created.")
print(adata)
print("\nDataset summary:")
display(dataset_summary)
print("\nRegion counts:")
display(region_counts)
print("\nSeed gene mean counts:")
display(expr_summary)

Synthetic prototype dataset created.
AnnData object with n_obs × n_vars = 1900 × 161
    obs: 'region', 'resistant_score', 'resistant_like', 'in_tissue'
    obsm: 'spatial'
    layers: 'counts'

Dataset summary:


,n_spots,n_genes,n_tumor,n_interface,n_stroma,x_min,x_max,y_min,y_max
0,1900,161,900,300,700,-59.664085,58.983471,-58.282463,59.15337



Region counts:


,region,n_spots
0,tumor,900
1,stroma,700
2,interface,300



Seed gene mean counts:


,gene,mean_count
0,CXCL12,1.477368
1,CXCR4,0.824210
2,TGFB1,1.709474
3,TGFBR2,0.883684
4,IFNG,0.224211
5,IFNGR1,0.341053


### Step 2

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [4]:
# CELL 3 — normalization and log transformation (keeping raw counts)

# Keep counts untouched
adata.layers["counts"] = adata.X.copy()

# Library-size normalization (CPM-like)
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform
sc.pp.log1p(adata)

# Save normalized matrix separately for safety
adata.layers["lognorm"] = adata.X.copy()

# Quick sanity checks
norm_summary = pd.DataFrame({
    "metric": [
        "mean_library_size_before",
        "mean_library_size_after",
        "matrix_min",
        "matrix_max"
    ],
    "value": [
        float(np.mean(np.array(adata.layers["counts"].sum(axis=1)).flatten())),
        float(np.mean(np.array(adata.X.sum(axis=1)).flatten())),
        float(adata.X.min()),
        float(adata.X.max())
    ]
})

if CONFIG["save_tables"]:
    norm_summary.to_csv(TABLES / "normalization_summary.csv", index=False)

print("Normalization complete.")
display(norm_summary)

Normalization complete.


,metric,value
0,mean_library_size_before,52.446842
1,mean_library_size_after,191.953201
2,matrix_min,0.000000
3,matrix_max,7.775676


### Step 3

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [6]:
# CELL 4 — construct the real spatial kNN graph from coordinates

coords = np.asarray(adata.obsm["spatial"], dtype=np.float32)
n_nodes = coords.shape[0]
k = CONFIG["graph_k"]

# sklearn NearestNeighbors includes the point itself as the first neighbor,
# so request k+1 and then drop self.
nbrs = NearestNeighbors(n_neighbors=k + 1, metric="euclidean")
nbrs.fit(coords)
distances, indices = nbrs.kneighbors(coords)

src, dst, edge_dist = [], [], []

for i in range(n_nodes):
    neigh_idx = indices[i, 1:]      # drop self
    neigh_dist = distances[i, 1:]   # drop self-distance
    for j, d in zip(neigh_idx, neigh_dist):
        src.append(i)
        dst.append(int(j))
        edge_dist.append(float(d))

edge_index = torch.tensor([src, dst], dtype=torch.long)
edge_distance = np.array(edge_dist, dtype=np.float32)

# Degree summary
deg_out = degree(edge_index[0], num_nodes=n_nodes).cpu().numpy()
deg_in = degree(edge_index[1], num_nodes=n_nodes).cpu().numpy()

graph_summary = pd.DataFrame([{
    "world": "true",
    "n_nodes": int(n_nodes),
    "n_edges_directed": int(edge_index.shape[1]),
    "k_requested": int(k),
    "mean_out_degree": float(deg_out.mean()),
    "min_out_degree": float(deg_out.min()),
    "max_out_degree": float(deg_out.max()),
    "mean_in_degree": float(deg_in.mean()),
    "min_in_degree": float(deg_in.min()),
    "max_in_degree": float(deg_in.max()),
    "mean_edge_distance": float(edge_distance.mean()),
    "median_edge_distance": float(np.median(edge_distance)),
    "max_edge_distance": float(edge_distance.max()),
}])

# Save graph artifacts for later cells
graph_artifacts = {
    "coords": coords,
    "edge_index_true": edge_index,
    "edge_distance_true": edge_distance,
}

if CONFIG["save_tables"]:
    graph_summary.to_csv(TABLES / "graph_summary_true.csv", index=False)

print("Real spatial kNN graph constructed.")
display(graph_summary)
print(f"edge_index shape: {tuple(edge_index.shape)}")

Real spatial kNN graph constructed.


,world,n_nodes,n_edges_directed,k_requested,mean_out_degree,min_out_degree,max_out_degree,mean_in_degree,min_in_degree,max_in_degree,mean_edge_distance,median_edge_distance,max_edge_distance
0,true,1900,15200,8,8.0,8.0,8.0,8.0,2.0,14.0,2.592515,2.243887,10.541575


edge_index shape: (2, 15200)


### Step 4

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [7]:
# CELL 5 — build the spatial-null graph for falsification
# Strategy:
# - preserve node identities
# - preserve exact out-degree (= k for every node)
# - break real spatial organization by assigning neighbors from random distant nodes
# - avoid self-loops and duplicate outgoing neighbors per node

coords = graph_artifacts["coords"]
edge_index_true = graph_artifacts["edge_index_true"]
n_nodes = coords.shape[0]
k = CONFIG["graph_k"]

rng = np.random.default_rng(17)

# Pairwise distances for "far-neighbor" sampling
# We use a distance threshold so the null graph breaks local tissue geometry.
# Here: choose neighbors outside the 75th percentile of all kNN edge distances.
distance_threshold = float(np.quantile(graph_artifacts["edge_distance_true"], 0.75))

# Compute full distance matrix once for this synthetic dataset size
# (1900 x 1900 is acceptable)
diff_x = coords[:, 0][:, None] - coords[:, 0][None, :]
diff_y = coords[:, 1][:, None] - coords[:, 1][None, :]
dist_matrix = np.sqrt(diff_x**2 + diff_y**2)

src_null, dst_null, edge_dist_null = [], [], []

for i in range(n_nodes):
    # candidate nodes that are far away enough, excluding self
    candidates = np.where((dist_matrix[i] > distance_threshold) & (np.arange(n_nodes) != i))[0]

    # fallback if threshold is too strict for any node
    if len(candidates) < k:
        candidates = np.where(np.arange(n_nodes) != i)[0]

    chosen = rng.choice(candidates, size=k, replace=False)

    for j in chosen:
        src_null.append(i)
        dst_null.append(int(j))
        edge_dist_null.append(float(dist_matrix[i, j]))

edge_index_spatial_null = torch.tensor([src_null, dst_null], dtype=torch.long)
edge_distance_spatial_null = np.array(edge_dist_null, dtype=np.float32)

# Degree summary
deg_out_null = degree(edge_index_spatial_null[0], num_nodes=n_nodes).cpu().numpy()
deg_in_null = degree(edge_index_spatial_null[1], num_nodes=n_nodes).cpu().numpy()

graph_summary_spatial_null = pd.DataFrame([{
    "world": "spatial_null",
    "n_nodes": int(n_nodes),
    "n_edges_directed": int(edge_index_spatial_null.shape[1]),
    "k_requested": int(k),
    "mean_out_degree": float(deg_out_null.mean()),
    "min_out_degree": float(deg_out_null.min()),
    "max_out_degree": float(deg_out_null.max()),
    "mean_in_degree": float(deg_in_null.mean()),
    "min_in_degree": float(deg_in_null.min()),
    "max_in_degree": float(deg_in_null.max()),
    "mean_edge_distance": float(edge_distance_spatial_null.mean()),
    "median_edge_distance": float(np.median(edge_distance_spatial_null)),
    "max_edge_distance": float(edge_distance_spatial_null.max()),
    "distance_threshold_used": float(distance_threshold),
}])

# Compare with the true graph
graph_comparison = pd.DataFrame([{
    "metric": "mean_edge_distance",
    "true": float(graph_artifacts["edge_distance_true"].mean()),
    "spatial_null": float(edge_distance_spatial_null.mean()),
}, {
    "metric": "median_edge_distance",
    "true": float(np.median(graph_artifacts["edge_distance_true"])),
    "spatial_null": float(np.median(edge_distance_spatial_null)),
}, {
    "metric": "max_edge_distance",
    "true": float(graph_artifacts["edge_distance_true"].max()),
    "spatial_null": float(edge_distance_spatial_null.max()),
}, {
    "metric": "mean_in_degree",
    "true": float(degree(edge_index_true[1], num_nodes=n_nodes).cpu().numpy().mean()),
    "spatial_null": float(deg_in_null.mean()),
}, {
    "metric": "mean_out_degree",
    "true": float(degree(edge_index_true[0], num_nodes=n_nodes).cpu().numpy().mean()),
    "spatial_null": float(deg_out_null.mean()),
}])

# Save artifacts
graph_artifacts["edge_index_spatial_null"] = edge_index_spatial_null
graph_artifacts["edge_distance_spatial_null"] = edge_distance_spatial_null
graph_artifacts["distance_threshold_spatial_null"] = distance_threshold

if CONFIG["save_tables"]:
    graph_summary_spatial_null.to_csv(TABLES / "graph_summary_spatial_null.csv", index=False)
    graph_comparison.to_csv(TABLES / "graph_comparison_true_vs_spatial_null.csv", index=False)

print("Spatial-null graph constructed.")
display(graph_summary_spatial_null)
print("\nTrue vs spatial-null comparison:")
display(graph_comparison)
print(f"edge_index_spatial_null shape: {tuple(edge_index_spatial_null.shape)}")

Spatial-null graph constructed.


,world,n_nodes,n_edges_directed,k_requested,mean_out_degree,min_out_degree,max_out_degree,mean_in_degree,min_in_degree,max_in_degree,mean_edge_distance,median_edge_distance,max_edge_distance,distance_threshold_used
0,spatial_null,1900,15200,8,8.0,8.0,8.0,8.0,0.0,18.0,42.869316,40.412109,116.145691,3.458034



True vs spatial-null comparison:


,metric,true,spatial_null
0,mean_edge_distance,2.592515,42.869316
1,median_edge_distance,2.243887,40.412109
2,max_edge_distance,10.541575,116.145691
3,mean_in_degree,8.000000,8.000000
4,mean_out_degree,8.000000,8.000000


edge_index_spatial_null shape: (2, 15200)


### Step 5

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [8]:
# CELL 6 — define pathways, build the true gene→pathway mask, and build the shuffled mask
# This cell creates:
# - pathway_registry
# - pathway_mask_true_df
# - pathway_mask_shuffled_df
# - pathway_masks dict for later model construction

# ---------------------------
# Pathway registry
# ---------------------------
pathway_registry = {
    "TGFB": ["TGFB1", "TGFBR2", "SMAD2", "SMAD3", "SMAD4", "SERPINE1", "COL1A1", "COL1A2"],
    "WNT": ["CTNNB1", "FZD7", "WNT5A", "AXIN2", "MYC", "CCND1"],
    "IFNG": ["IFNG", "IFNGR1", "STAT1", "IRF1", "JAK1", "JAK2", "GBP1", "CXCL9"],
    "Antigen_Presentation": ["HLA-A", "HLA-B", "B2M", "TAP1", "PSMB8", "NLRC5"],
}

available_genes = set(adata.var_names.tolist())

# Keep only genes present in adata
pathway_registry_filtered = {}
missing_by_pathway = {}

for pathway, genes in pathway_registry.items():
    present = [g for g in genes if g in available_genes]
    missing = [g for g in genes if g not in available_genes]
    pathway_registry_filtered[pathway] = present
    missing_by_pathway[pathway] = missing

pathway_names = list(pathway_registry_filtered.keys())
n_pathways = len(pathway_names)
n_genes = adata.n_vars

gene_index = {g: i for i, g in enumerate(adata.var_names.tolist())}

# ---------------------------
# True mask
# rows = genes, cols = pathways
# ---------------------------
mask_true = np.zeros((n_genes, n_pathways), dtype=np.float32)

for p_idx, pathway in enumerate(pathway_names):
    for gene in pathway_registry_filtered[pathway]:
        mask_true[gene_index[gene], p_idx] = 1.0

pathway_mask_true_df = pd.DataFrame(mask_true, index=adata.var_names, columns=pathway_names)

# ---------------------------
# Shuffled mask
# preserve pathway sizes, break biology
# ---------------------------
rng = np.random.default_rng(17)
mask_shuffled = np.zeros((n_genes, n_pathways), dtype=np.float32)

all_gene_indices = np.arange(n_genes)

for p_idx, pathway in enumerate(pathway_names):
    pathway_size = int(mask_true[:, p_idx].sum())

    # sample genes uniformly from all genes, without replacement within pathway
    chosen = rng.choice(all_gene_indices, size=pathway_size, replace=False)
    mask_shuffled[chosen, p_idx] = 1.0

pathway_mask_shuffled_df = pd.DataFrame(mask_shuffled, index=adata.var_names, columns=pathway_names)

# ---------------------------
# Sanity summaries
# ---------------------------
pathway_summary_rows = []

for pathway in pathway_names:
    true_genes = pathway_mask_true_df.index[pathway_mask_true_df[pathway] == 1].tolist()
    shuf_genes = pathway_mask_shuffled_df.index[pathway_mask_shuffled_df[pathway] == 1].tolist()

    overlap = len(set(true_genes).intersection(set(shuf_genes)))
    pathway_summary_rows.append({
        "pathway": pathway,
        "n_true_genes": len(true_genes),
        "n_shuffled_genes": len(shuf_genes),
        "overlap_true_vs_shuffled": overlap,
        "missing_original_genes": ", ".join(missing_by_pathway[pathway]) if missing_by_pathway[pathway] else "",
    })

pathway_summary = pd.DataFrame(pathway_summary_rows)

# Gene-level membership summary
gene_membership_summary = pd.DataFrame({
    "gene": adata.var_names.tolist(),
    "n_true_pathways": mask_true.sum(axis=1).astype(int),
    "n_shuffled_pathways": mask_shuffled.sum(axis=1).astype(int),
})

# Save artifacts for later cells
pathway_masks = {
    "true": torch.tensor(mask_true, dtype=torch.float32),
    "mask_shuffle": torch.tensor(mask_shuffled, dtype=torch.float32),
}

pathway_artifacts = {
    "pathway_names": pathway_names,
    "pathway_registry_filtered": pathway_registry_filtered,
    "missing_by_pathway": missing_by_pathway,
    "pathway_mask_true_df": pathway_mask_true_df,
    "pathway_mask_shuffled_df": pathway_mask_shuffled_df,
}

if CONFIG["save_tables"]:
    pathway_summary.to_csv(TABLES / "pathway_summary_true_vs_shuffled.csv", index=False)
    pathway_mask_true_df.to_csv(TABLES / "pathway_mask_true.csv")
    pathway_mask_shuffled_df.to_csv(TABLES / "pathway_mask_shuffled.csv")
    gene_membership_summary.to_csv(TABLES / "gene_membership_summary.csv", index=False)

print("True and shuffled pathway masks created.")
print(f"Number of pathways: {n_pathways}")
print(f"Number of genes: {n_genes}")
print("\nPathway summary:")
display(pathway_summary)
print("\nFirst 12 genes x pathways (true mask):")
display(pathway_mask_true_df.iloc[:12, :])
print("\nFirst 12 genes x pathways (shuffled mask):")
display(pathway_mask_shuffled_df.iloc[:12, :])

True and shuffled pathway masks created.
Number of pathways: 4
Number of genes: 161

Pathway summary:


,pathway,n_true_genes,n_shuffled_genes,overlap_true_vs_shuffled,missing_original_genes
0,TGFB,8,8,0,
1,WNT,6,6,1,
2,IFNG,8,8,1,
3,Antigen_Presentation,6,6,1,



First 12 genes x pathways (true mask):


,TGFB,WNT,IFNG,Antigen_Presentation
CXCL12,0.0,0.0,0.0,0.0
CXCR4,0.0,0.0,0.0,0.0
TGFB1,1.0,0.0,0.0,0.0
TGFBR2,1.0,0.0,0.0,0.0
IFNG,0.0,0.0,1.0,0.0
IFNGR1,0.0,0.0,1.0,0.0
SMAD2,1.0,0.0,0.0,0.0
SMAD3,1.0,0.0,0.0,0.0
SMAD4,1.0,0.0,0.0,0.0
SERPINE1,1.0,0.0,0.0,0.0



First 12 genes x pathways (shuffled mask):


,TGFB,WNT,IFNG,Antigen_Presentation
CXCL12,0.0,0.0,0.0,0.0
CXCR4,0.0,0.0,0.0,0.0
TGFB1,0.0,1.0,0.0,0.0
TGFBR2,0.0,0.0,0.0,0.0
IFNG,0.0,0.0,0.0,0.0
IFNGR1,0.0,0.0,1.0,0.0
SMAD2,0.0,0.0,0.0,0.0
SMAD3,0.0,1.0,0.0,0.0
SMAD4,0.0,0.0,0.0,0.0
SERPINE1,0.0,0.0,0.0,0.0


#### Output interpretation
- Compare results to falsification expectations
- Look for separation between TRUE vs NULL conditions
- Strong separation indicates meaningful signal usage


### Step 6

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [9]:
# CELL 7 — construct ligand-receptor candidate edge tables for true and spatial-null worlds,
# and define seeded + decoy axis registries for falsification

# This cell builds:
# - edge_table_true
# - edge_table_spatial_null
# - axis_registry (seeded + random decoys + expression-matched decoys)
#
# Important:
# We operationalize "communication" here only as putative LR-compatible adjacency
# based on expression over graph edges. This is not causal signaling.

# ---------------------------
# Helpers
# ---------------------------
expr = adata.layers["lognorm"] if "lognorm" in adata.layers else adata.X
if sparse.issparse(expr):
    expr = expr.toarray()
expr = np.asarray(expr, dtype=np.float32)

gene_names = adata.var_names.tolist()
gene_to_idx = {g: i for i, g in enumerate(gene_names)}

seeded_axes = [tuple(x) for x in CONFIG["seeded_axes"]]

def percentile_threshold(values: np.ndarray, q: float = 0.75) -> float:
    values = np.asarray(values, dtype=np.float32)
    return float(np.quantile(values, q))

def compute_gene_thresholds(expr_matrix: np.ndarray, genes: list[str], q: float = 0.75) -> dict:
    thresholds = {}
    for g in genes:
        idx = gene_to_idx[g]
        thresholds[g] = percentile_threshold(expr_matrix[:, idx], q=q)
    return thresholds

def build_axis_edge_table(
    expr_matrix: np.ndarray,
    edge_index_tensor: torch.Tensor,
    axes: list[tuple[str, str]],
    thresholds: dict,
    world_name: str,
) -> pd.DataFrame:
    src = edge_index_tensor[0].cpu().numpy()
    dst = edge_index_tensor[1].cpu().numpy()

    rows = []
    for ligand, receptor in axes:
        l_idx = gene_to_idx[ligand]
        r_idx = gene_to_idx[receptor]
        l_thr = thresholds[ligand]
        r_thr = thresholds[receptor]

        l_expr_src = expr_matrix[src, l_idx]
        r_expr_dst = expr_matrix[dst, r_idx]

        active = (l_expr_src > l_thr) & (r_expr_dst > r_thr)

        if active.sum() == 0:
            rows.append({
                "world": world_name,
                "ligand": ligand,
                "receptor": receptor,
                "axis": f"{ligand}->{receptor}",
                "n_active_edges": 0,
                "active_edge_fraction": 0.0,
                "mean_ligand_src_expr_active": 0.0,
                "mean_receptor_dst_expr_active": 0.0,
                "mean_joint_score_active": 0.0,
            })
        else:
            joint_score = l_expr_src[active] * r_expr_dst[active]
            rows.append({
                "world": world_name,
                "ligand": ligand,
                "receptor": receptor,
                "axis": f"{ligand}->{receptor}",
                "n_active_edges": int(active.sum()),
                "active_edge_fraction": float(active.mean()),
                "mean_ligand_src_expr_active": float(l_expr_src[active].mean()),
                "mean_receptor_dst_expr_active": float(r_expr_dst[active].mean()),
                "mean_joint_score_active": float(joint_score.mean()),
            })

    return pd.DataFrame(rows)

# ---------------------------
# Decoy construction
# ---------------------------
all_seed_genes = sorted(set([g for pair in seeded_axes for g in pair]))

# candidate ligands/receptors from genes not in seeded genes
candidate_genes = [g for g in gene_names if g not in all_seed_genes]

# mean expression for matching
gene_mean_expr = pd.Series(expr.mean(axis=0), index=gene_names)

def sample_random_decoys(
    candidate_gene_list: list[str],
    n_decoys: int,
    seeded_axis_set: set[tuple[str, str]],
    rng_seed: int = 17,
) -> list[tuple[str, str]]:
    rng_local = np.random.default_rng(rng_seed)
    decoys = set()
    max_tries = 10000
    tries = 0

    while len(decoys) < n_decoys and tries < max_tries:
        l = str(rng_local.choice(candidate_gene_list))
        r = str(rng_local.choice(candidate_gene_list))
        pair = (l, r)
        if l != r and pair not in seeded_axis_set:
            decoys.add(pair)
        tries += 1

    return sorted(list(decoys))

def find_expr_matched_gene(target_gene: str, candidate_gene_list: list[str], used: set[str]) -> str:
    target_expr = float(gene_mean_expr[target_gene])
    ranked = sorted(
        [g for g in candidate_gene_list if g not in used],
        key=lambda g: abs(float(gene_mean_expr[g]) - target_expr)
    )
    if len(ranked) == 0:
        raise ValueError(f"No expression-matched candidate available for target gene {target_gene}")
    return ranked[0]

def sample_expression_matched_decoys(
    seeded_axes_list: list[tuple[str, str]],
    candidate_gene_list: list[str],
) -> list[tuple[str, str]]:
    used = set()
    decoys = []

    for ligand, receptor in seeded_axes_list:
        l_match = find_expr_matched_gene(ligand, candidate_gene_list, used)
        used.add(l_match)
        r_match = find_expr_matched_gene(receptor, candidate_gene_list, used)
        used.add(r_match)
        decoys.append((l_match, r_match))

    return decoys

seeded_axis_set = set(seeded_axes)

random_decoys = sample_random_decoys(
    candidate_gene_list=candidate_genes,
    n_decoys=CONFIG["num_random_decoys"],
    seeded_axis_set=seeded_axis_set,
    rng_seed=17,
)

expr_matched_decoys = sample_expression_matched_decoys(
    seeded_axes_list=seeded_axes,
    candidate_gene_list=candidate_genes,
)

# Axis registry
axis_registry_rows = []
for ligand, receptor in seeded_axes:
    axis_registry_rows.append({
        "axis": f"{ligand}->{receptor}",
        "ligand": ligand,
        "receptor": receptor,
        "axis_type": "seeded",
    })

for ligand, receptor in random_decoys:
    axis_registry_rows.append({
        "axis": f"{ligand}->{receptor}",
        "ligand": ligand,
        "receptor": receptor,
        "axis_type": "random_decoy",
    })

for ligand, receptor in expr_matched_decoys:
    axis_registry_rows.append({
        "axis": f"{ligand}->{receptor}",
        "ligand": ligand,
        "receptor": receptor,
        "axis_type": "expr_matched_decoy",
    })

axis_registry = pd.DataFrame(axis_registry_rows)

# ---------------------------
# Expression thresholds
# ---------------------------
all_axis_genes = sorted(set(axis_registry["ligand"]).union(set(axis_registry["receptor"])))
gene_thresholds = compute_gene_thresholds(expr, all_axis_genes, q=0.75)

gene_thresholds_df = pd.DataFrame({
    "gene": list(gene_thresholds.keys()),
    "threshold_q75": list(gene_thresholds.values()),
    "mean_expr": [float(gene_mean_expr[g]) for g in gene_thresholds.keys()],
}).sort_values("gene").reset_index(drop=True)

# ---------------------------
# Edge tables for both worlds
# ---------------------------
all_axes = list(zip(axis_registry["ligand"], axis_registry["receptor"]))

edge_table_true = build_axis_edge_table(
    expr_matrix=expr,
    edge_index_tensor=graph_artifacts["edge_index_true"],
    axes=all_axes,
    thresholds=gene_thresholds,
    world_name="true",
)

edge_table_spatial_null = build_axis_edge_table(
    expr_matrix=expr,
    edge_index_tensor=graph_artifacts["edge_index_spatial_null"],
    axes=all_axes,
    thresholds=gene_thresholds,
    world_name="spatial_null",
)

edge_table_true = edge_table_true.merge(axis_registry, on=["axis", "ligand", "receptor"], how="left")
edge_table_spatial_null = edge_table_spatial_null.merge(axis_registry, on=["axis", "ligand", "receptor"], how="left")

# Seeded summary preview
edge_preview = pd.concat([
    edge_table_true,
    edge_table_spatial_null,
], axis=0, ignore_index=True)

edge_preview_seeded = edge_preview[edge_preview["axis_type"] == "seeded"].copy()
edge_preview_seeded = edge_preview_seeded.sort_values(["world", "axis"]).reset_index(drop=True)

# Save artifacts
edge_artifacts = {
    "edge_table_true": edge_table_true,
    "edge_table_spatial_null": edge_table_spatial_null,
    "axis_registry": axis_registry,
    "gene_thresholds": gene_thresholds,
}

if CONFIG["save_tables"]:
    axis_registry.to_csv(TABLES / "axis_registry.csv", index=False)
    gene_thresholds_df.to_csv(TABLES / "axis_gene_thresholds.csv", index=False)
    edge_table_true.to_csv(TABLES / "edge_table_true.csv", index=False)
    edge_table_spatial_null.to_csv(TABLES / "edge_table_spatial_null.csv", index=False)

print("Axis registry and candidate LR edge tables created.")
print(f"Total axes: {len(axis_registry)}")
print(f"  Seeded axes: {(axis_registry['axis_type'] == 'seeded').sum()}")
print(f"  Random decoys: {(axis_registry['axis_type'] == 'random_decoy').sum()}")
print(f"  Expression-matched decoys: {(axis_registry['axis_type'] == 'expr_matched_decoy').sum()}")

print("\nAxis registry preview:")
display(axis_registry.head(12))

print("\nSeeded axis edge summary by world:")
display(edge_preview_seeded)

print("\nGene threshold preview:")
display(gene_thresholds_df.head(12))

Axis registry and candidate LR edge tables created.
Total axes: 31
  Seeded axes: 3
  Random decoys: 25
  Expression-matched decoys: 3

Axis registry preview:


,axis,ligand,receptor,axis_type
0,CXCL12->CXCR4,CXCL12,CXCR4,seeded
1,TGFB1->TGFBR2,TGFB1,TGFBR2,seeded
2,IFNG->IFNGR1,IFNG,IFNGR1,seeded
3,BG_GENE_022->BG_GENE_059,BG_GENE_022,BG_GENE_059,random_decoy
4,BG_GENE_029->BG_GENE_053,BG_GENE_029,BG_GENE_053,random_decoy
5,BG_GENE_030->COL1A2,BG_GENE_030,COL1A2,random_decoy
6,BG_GENE_036->BG_GENE_025,BG_GENE_036,BG_GENE_025,random_decoy
7,BG_GENE_036->BG_GENE_052,BG_GENE_036,BG_GENE_052,random_decoy
8,BG_GENE_040->KRT8,BG_GENE_040,KRT8,random_decoy
9,BG_GENE_041->SMAD3,BG_GENE_041,SMAD3,random_decoy



Seeded axis edge summary by world:


,world,ligand,receptor,axis,n_active_edges,active_edge_fraction,mean_ligand_src_expr_active,mean_receptor_dst_expr_active,mean_joint_score_active,axis_type
0,spatial_null,CXCL12,CXCR4,CXCL12->CXCR4,909,0.059803,6.658283,6.078218,40.473831,seeded
1,spatial_null,IFNG,IFNGR1,IFNG->IFNGR1,771,0.050724,5.251700,5.445627,28.608641,seeded
2,spatial_null,TGFB1,TGFBR2,TGFB1->TGFBR2,942,0.061974,6.648264,6.127302,40.737885,seeded
3,true,CXCL12,CXCR4,CXCL12->CXCR4,730,0.048026,6.618584,6.067761,40.154732,seeded
4,true,IFNG,IFNGR1,IFNG->IFNGR1,788,0.051842,5.245601,5.453158,28.609142,seeded
5,true,TGFB1,TGFBR2,TGFB1->TGFBR2,1259,0.082829,6.664135,6.170126,41.118393,seeded



Gene threshold preview:


,gene,threshold_q75,mean_expr
0,BG_GENE_005,0.0,0.816524
1,BG_GENE_022,0.0,0.743507
2,BG_GENE_023,0.0,0.810177
3,BG_GENE_025,0.0,0.813430
4,BG_GENE_029,0.0,0.720782
5,BG_GENE_030,0.0,0.766586
6,BG_GENE_032,0.0,0.784551
7,BG_GENE_036,0.0,0.859049
8,BG_GENE_037,0.0,0.731616
9,BG_GENE_040,0.0,0.657593


### Step 7

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [10]:
# CELL 8 — package node features, labels, and PyG graph objects for each falsification world

# This cell creates:
# - node feature matrix from the normalized expression data
# - binary target label from the designed latent state (resistant_like vs other)
# - train/val/test masks with fixed seeds
# - PyG Data objects for:
#       true         = real graph + true pathway mask
#       mask_shuffle = real graph + shuffled pathway mask
#       spatial_null = null graph + true pathway mask
#
# Notes:
# - We keep the task deliberately simple: predict resistant_like state
# - This is enough for the kill-test because we mainly care about
#   pathway alignment, seeded-axis ranking, and spatial dependence

# ---------------------------
# Node features
# ---------------------------
X = adata.layers["lognorm"] if "lognorm" in adata.layers else adata.X
if sparse.issparse(X):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)

# Optional feature scaling across genes
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Binary labels: resistant_like = 1, other = 0
y = (adata.obs["resistant_like"].astype(str).values == "resistant_like").astype(np.int64)

# Region ids for later summaries/diagnostics
region_categories = ["tumor", "interface", "stroma"]
region_to_id = {r: i for i, r in enumerate(region_categories)}
region_id = np.array([region_to_id[r] for r in adata.obs["region"].astype(str).values], dtype=np.int64)

# ---------------------------
# Reproducible train/val/test splits
# ---------------------------
def make_split_masks(n: int, seed: int, train_frac: float = 0.70, val_frac: float = 0.15):
    rng_local = np.random.default_rng(seed)
    idx = np.arange(n)
    rng_local.shuffle(idx)

    n_train = int(round(train_frac * n))
    n_val = int(round(val_frac * n))
    n_test = n - n_train - n_val

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train + n_val]
    test_idx = idx[n_train + n_val:]

    train_mask = np.zeros(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)
    test_mask = np.zeros(n, dtype=bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    return train_mask, val_mask, test_mask

split_registry_rows = []
split_masks_by_seed = {}

for seed in CONFIG["seeds"]:
    train_mask, val_mask, test_mask = make_split_masks(n=adata.n_obs, seed=seed)
    split_masks_by_seed[seed] = {
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
    }
    split_registry_rows.append({
        "seed": seed,
        "n_train": int(train_mask.sum()),
        "n_val": int(val_mask.sum()),
        "n_test": int(test_mask.sum()),
        "train_pos_frac": float(y[train_mask].mean()),
        "val_pos_frac": float(y[val_mask].mean()),
        "test_pos_frac": float(y[test_mask].mean()),
    })

split_registry = pd.DataFrame(split_registry_rows)

# ---------------------------
# Helper: build PyG Data object
# ---------------------------
def build_pyg_data(
    x_array: np.ndarray,
    y_array: np.ndarray,
    edge_index_tensor: torch.Tensor,
    coords_array: np.ndarray,
    region_array: np.ndarray,
    pathway_mask_tensor: torch.Tensor,
    world_name: str,
) -> Data:
    data = Data(
        x=torch.tensor(x_array, dtype=torch.float32),
        y=torch.tensor(y_array, dtype=torch.long),
        edge_index=edge_index_tensor.clone().detach(),
    )
    data.coords = torch.tensor(coords_array, dtype=torch.float32)
    data.region_id = torch.tensor(region_array, dtype=torch.long)
    data.pathway_mask = pathway_mask_tensor.clone().detach()
    data.world = world_name
    return data

# ---------------------------
# World-specific graph objects
# ---------------------------
data_true = build_pyg_data(
    x_array=X_scaled,
    y_array=y,
    edge_index_tensor=graph_artifacts["edge_index_true"],
    coords_array=graph_artifacts["coords"],
    region_array=region_id,
    pathway_mask_tensor=pathway_masks["true"],
    world_name="true",
)

data_mask_shuffle = build_pyg_data(
    x_array=X_scaled,
    y_array=y,
    edge_index_tensor=graph_artifacts["edge_index_true"],
    coords_array=graph_artifacts["coords"],
    region_array=region_id,
    pathway_mask_tensor=pathway_masks["mask_shuffle"],
    world_name="mask_shuffle",
)

data_spatial_null = build_pyg_data(
    x_array=X_scaled,
    y_array=y,
    edge_index_tensor=graph_artifacts["edge_index_spatial_null"],
    coords_array=graph_artifacts["coords"],
    region_array=region_id,
    pathway_mask_tensor=pathway_masks["true"],
    world_name="spatial_null",
)

world_data = {
    "true": data_true,
    "mask_shuffle": data_mask_shuffle,
    "spatial_null": data_spatial_null,
}

# ---------------------------
# Summaries
# ---------------------------
label_summary = pd.DataFrame([{
    "n_nodes": int(len(y)),
    "n_positive_resistant_like": int(y.sum()),
    "positive_fraction": float(y.mean()),
    "n_negative_other": int((y == 0).sum()),
}])

feature_summary = pd.DataFrame([{
    "n_features": int(X_scaled.shape[1]),
    "feature_mean_global": float(X_scaled.mean()),
    "feature_std_global": float(X_scaled.std()),
    "feature_min": float(X_scaled.min()),
    "feature_max": float(X_scaled.max()),
}])

world_summary_rows = []
for world_name, data_obj in world_data.items():
    world_summary_rows.append({
        "world": world_name,
        "n_nodes": int(data_obj.num_nodes),
        "n_edges": int(data_obj.edge_index.shape[1]),
        "n_features": int(data_obj.x.shape[1]),
        "n_pathways": int(data_obj.pathway_mask.shape[1]),
        "uses_true_graph": bool(world_name != "spatial_null"),
        "uses_true_mask": bool(world_name != "mask_shuffle"),
    })

world_summary = pd.DataFrame(world_summary_rows)

# Save artifacts
dataset_artifacts = {
    "X_scaled": X_scaled,
    "y": y,
    "region_id": region_id,
    "region_to_id": region_to_id,
    "split_masks_by_seed": split_masks_by_seed,
    "world_data": world_data,
}

if CONFIG["save_tables"]:
    split_registry.to_csv(TABLES / "split_registry.csv", index=False)
    label_summary.to_csv(TABLES / "label_summary.csv", index=False)
    feature_summary.to_csv(TABLES / "feature_summary.csv", index=False)
    world_summary.to_csv(TABLES / "world_summary.csv", index=False)

print("World datasets packaged successfully.")
print("\nLabel summary:")
display(label_summary)

print("\nFeature summary:")
display(feature_summary)

print("\nSplit registry:")
display(split_registry)

print("\nWorld summary:")
display(world_summary)

World datasets packaged successfully.

Label summary:


,n_nodes,n_positive_resistant_like,positive_fraction,n_negative_other
0,1900,760,0.4,1140



Feature summary:


,n_features,feature_mean_global,feature_std_global,feature_min,feature_max
0,161,6.983428e-10,1.0,-1.239407,3.71163



Split registry:


,seed,n_train,n_val,n_test,train_pos_frac,val_pos_frac,test_pos_frac
0,11,1330,285,285,0.391729,0.445614,0.392982
1,17,1330,285,285,0.405263,0.350877,0.424561
2,23,1330,285,285,0.412782,0.347368,0.392982



World summary:


,world,n_nodes,n_edges,n_features,n_pathways,uses_true_graph,uses_true_mask
0,true,1900,15200,161,4,True,True
1,mask_shuffle,1900,15200,161,4,True,False
2,spatial_null,1900,15200,161,4,False,True


### Step 8

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [11]:
# CELL 9 — define the SpatialMMKPNN prototype model and forward-pass helpers
# Architecture:
#   expression features -> GAT encoder -> gene logits -> masked pathway bottleneck -> classifier
#
# Why this form:
# - keeps the concept bottleneck explicit
# - lets us test mask dependence directly
# - keeps training simple for the kill-test

class SpatialMMKPNN(nn.Module):
    def __init__(
        self,
        n_input_genes: int,
        hidden_dim: int,
        n_pathways: int,
        pathway_mask: torch.Tensor,
        gat_heads: int = 4,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.n_input_genes = n_input_genes
        self.hidden_dim = hidden_dim
        self.n_pathways = n_pathways
        self.gat_heads = gat_heads
        self.dropout = dropout

        # GAT encoder
        self.gat1 = GATConv(
            in_channels=n_input_genes,
            out_channels=hidden_dim,
            heads=gat_heads,
            concat=True,
            dropout=dropout,
        )
        self.gat2 = GATConv(
            in_channels=hidden_dim * gat_heads,
            out_channels=hidden_dim,
            heads=1,
            concat=True,
            dropout=dropout,
        )

        # Gene-level projection before pathway bottleneck
        self.gene_proj = nn.Linear(hidden_dim, n_input_genes)

        # Masked pathway bottleneck
        # pathway_weight has shape [n_genes, n_pathways]
        self.pathway_weight = nn.Parameter(torch.randn(n_input_genes, n_pathways) * 0.02)
        self.pathway_bias = nn.Parameter(torch.zeros(n_pathways))

        # Fixed binary mask (not trainable)
        self.register_buffer("pathway_mask", pathway_mask.clone().float())

        # Final classifier from pathway concepts
        self.classifier = nn.Sequential(
            nn.Linear(n_pathways, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, data: Data):
        x, edge_index = data.x, data.edge_index

        h1 = self.gat1(x, edge_index)
        h1 = F.elu(h1)
        h1 = F.dropout(h1, p=self.dropout, training=self.training)

        h2 = self.gat2(h1, edge_index)
        h2 = F.elu(h2)
        h2 = F.dropout(h2, p=self.dropout, training=self.training)

        gene_logits = self.gene_proj(h2)

        # Apply masked pathway projection
        masked_weight = self.pathway_weight * self.pathway_mask
        pathway_scores = gene_logits @ masked_weight + self.pathway_bias
        pathway_scores = F.relu(pathway_scores)

        logits = self.classifier(pathway_scores)

        return {
            "logits": logits,
            "embedding": h2,
            "gene_logits": gene_logits,
            "pathway_scores": pathway_scores,
        }


def make_model_for_world(world_name: str, data_obj: Data) -> SpatialMMKPNN:
    model = SpatialMMKPNN(
        n_input_genes=data_obj.x.shape[1],
        hidden_dim=CONFIG["hidden_dim"],
        n_pathways=data_obj.pathway_mask.shape[1],
        pathway_mask=data_obj.pathway_mask,
        gat_heads=CONFIG["gat_heads"],
        dropout=CONFIG["dropout"],
    )
    return model.to(DEVICE)


def count_parameters(model: nn.Module) -> pd.DataFrame:
    rows = []
    total = 0
    trainable = 0

    for name, param in model.named_parameters():
        n = param.numel()
        total += n
        if param.requires_grad:
            trainable += n
        rows.append({
            "parameter": name,
            "shape": tuple(param.shape),
            "n_params": int(n),
            "trainable": bool(param.requires_grad),
        })

    summary_df = pd.DataFrame(rows)
    total_df = pd.DataFrame([{
        "total_params": int(total),
        "trainable_params": int(trainable),
        "frozen_params": int(total - trainable),
    }])

    return summary_df, total_df


# Instantiate one model per world for sanity check
model_registry = {}
model_param_tables = {}
model_total_tables = []

for world_name, data_obj in dataset_artifacts["world_data"].items():
    model = make_model_for_world(world_name, data_obj)
    model_registry[world_name] = model

    param_df, total_df = count_parameters(model)
    param_df["world"] = world_name
    total_df["world"] = world_name

    model_param_tables[world_name] = param_df
    model_total_tables.append(total_df)

model_total_summary = pd.concat(model_total_tables, axis=0, ignore_index=True)

if CONFIG["save_tables"]:
    model_total_summary.to_csv(TABLES / "model_total_summary.csv", index=False)
    for world_name, param_df in model_param_tables.items():
        param_df.to_csv(TABLES / f"model_params_{world_name}.csv", index=False)

print("Model definitions completed.")
print("\nModel total summary:")
display(model_total_summary)

print("\nParameter preview for TRUE world:")
display(model_param_tables["true"].head(12))

Model definitions completed.

Model total summary:


,total_params,trainable_params,frozen_params,world
0,70123,70123,0,true
1,70123,70123,0,mask_shuffle
2,70123,70123,0,spatial_null



Parameter preview for TRUE world:


,parameter,shape,n_params,trainable,world
0,pathway_weight,"(161, 4)",644,True,true
1,pathway_bias,"(4,)",4,True,true
2,gat1.att_src,"(1, 4, 64)",256,True,true
3,gat1.att_dst,"(1, 4, 64)",256,True,true
4,gat1.bias,"(256,)",256,True,true
5,gat1.lin.weight,"(256, 161)",41216,True,true
6,gat2.att_src,"(1, 1, 64)",64,True,true
7,gat2.att_dst,"(1, 1, 64)",64,True,true
8,gat2.bias,"(64,)",64,True,true
9,gat2.lin.weight,"(64, 256)",16384,True,true


### Step 9

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [12]:
# CELL 10 — training loop and evaluation helpers for one run
# This cell defines:
# - training function
# - prediction metrics
# - pathway alignment metric
# - seeded-axis ranking metric
# - interface-attention proxy metric
#
# Important:
# For this first implementation, "attention/interface score" is approximated from
# node-level pathway activity and edge-level LR-compatible scores, because extracting
# per-edge attention weights robustly across PyG versions is often messy.
# We will use a graph- and biology-sensitive proxy that is enough for falsification.

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ---------------------------
# Utility helpers
# ---------------------------
def move_data_to_device(data_obj: Data, device: torch.device) -> Data:
    data_dev = Data(
        x=data_obj.x.to(device),
        y=data_obj.y.to(device),
        edge_index=data_obj.edge_index.to(device),
    )
    data_dev.coords = data_obj.coords.to(device)
    data_dev.region_id = data_obj.region_id.to(device)
    data_dev.pathway_mask = data_obj.pathway_mask.to(device)
    data_dev.world = data_obj.world
    return data_dev


def tensor_to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def compute_classification_metrics(y_true, y_prob, y_pred) -> dict:
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }
    # ROC AUC can fail if only one class is present in split
    try:
        out["roc_auc"] = float(roc_auc_score(y_true, y_prob))
    except Exception:
        out["roc_auc"] = np.nan
    return out


# ---------------------------
# Pathway alignment metric
# ---------------------------
# expected:
#   TGFB up in resistant_like
#   WNT up in resistant_like
#   IFNG down in resistant_like
#   Antigen_Presentation down in resistant_like
EXPECTED_DIRECTIONS = CONFIG["expected_pathway_direction"]

def compute_pathway_alignment(pathway_scores: np.ndarray, y_true_binary: np.ndarray, pathway_names: list[str]) -> pd.DataFrame:
    rows = []
    resistant_mask = y_true_binary == 1
    other_mask = y_true_binary == 0

    for j, pathway in enumerate(pathway_names):
        mean_res = float(pathway_scores[resistant_mask, j].mean())
        mean_other = float(pathway_scores[other_mask, j].mean())
        delta = mean_res - mean_other

        expected = EXPECTED_DIRECTIONS[pathway]
        observed_direction = "up" if delta > 0 else "down" if delta < 0 else "flat"
        sign_match = (
            (expected == "up" and delta > 0) or
            (expected == "down" and delta < 0)
        )

        rows.append({
            "pathway": pathway,
            "mean_resistant_like": mean_res,
            "mean_other": mean_other,
            "delta_resistant_minus_other": delta,
            "expected_direction": expected,
            "observed_direction": observed_direction,
            "sign_match": bool(sign_match),
            "abs_delta": abs(delta),
        })

    return pd.DataFrame(rows)


def summarize_pathway_alignment(pathway_alignment_df: pd.DataFrame) -> dict:
    sign_match_rate = float(pathway_alignment_df["sign_match"].mean())

    # ranked importance by absolute delta
    ranked = pathway_alignment_df.sort_values("abs_delta", ascending=False)["pathway"].tolist()
    expected_top = set(["TGFB", "WNT", "IFNG", "Antigen_Presentation"])
    top4 = set(ranked[:4])
    top4_overlap = len(expected_top.intersection(top4))

    return {
        "pathway_sign_match_rate": sign_match_rate,
        "pathway_top4_overlap": int(top4_overlap),
    }


# ---------------------------
# Seeded-axis ranking metric
# ---------------------------
# Uses precomputed edge tables and ranks axes by mean_joint_score_active
def rank_axes_from_edge_table(edge_table_world: pd.DataFrame) -> pd.DataFrame:
    ranked = edge_table_world.sort_values(
        ["mean_joint_score_active", "n_active_edges"],
        ascending=[False, False]
    ).reset_index(drop=True)

    ranked["rank_joint_score"] = np.arange(1, len(ranked) + 1)
    return ranked


def summarize_seeded_axis_ranks(ranked_axes_df: pd.DataFrame) -> pd.DataFrame:
    seeded = ranked_axes_df[ranked_axes_df["axis_type"] == "seeded"].copy()
    seeded = seeded.sort_values("rank_joint_score").reset_index(drop=True)
    return seeded[[
        "axis", "axis_type", "n_active_edges", "active_edge_fraction",
        "mean_joint_score_active", "rank_joint_score"
    ]]


def compute_seed_vs_decoy_margin(ranked_axes_df: pd.DataFrame) -> dict:
    seeded_scores = ranked_axes_df.loc[ranked_axes_df["axis_type"] == "seeded", "mean_joint_score_active"].values
    decoy_scores = ranked_axes_df.loc[ranked_axes_df["axis_type"] != "seeded", "mean_joint_score_active"].values

    if len(seeded_scores) == 0 or len(decoy_scores) == 0:
        return {
            "seeded_mean_joint_score": np.nan,
            "decoy_mean_joint_score": np.nan,
            "seed_minus_decoy_margin": np.nan,
        }

    return {
        "seeded_mean_joint_score": float(np.mean(seeded_scores)),
        "decoy_mean_joint_score": float(np.mean(decoy_scores)),
        "seed_minus_decoy_margin": float(np.mean(seeded_scores) - np.mean(decoy_scores)),
    }


# ---------------------------
# Interface localization proxy
# ---------------------------
# We quantify whether high-scoring active edges touch the interface region.
# This is a falsification-friendly proxy for spatial localization.
def compute_interface_edge_score(
    edge_index_tensor: torch.Tensor,
    edge_table_world: pd.DataFrame,
    region_id_array: np.ndarray,
    axis_subset: list[str] | None = None,
) -> dict:
    # region ids: tumor=0, interface=1, stroma=2
    src = edge_index_tensor[0].cpu().numpy()
    dst = edge_index_tensor[1].cpu().numpy()

    # interface-touching if either endpoint is interface
    interface_touch = (region_id_array[src] == 1) | (region_id_array[dst] == 1)
    baseline_interface_fraction = float(interface_touch.mean())

    df = edge_table_world.copy()
    if axis_subset is not None:
        df = df[df["axis"].isin(axis_subset)].copy()

    # weighted proxy from top active axes
    # We do not have edge-resolved per-axis masks stored yet, so use axis-level strength
    # together with graph interface baseline. This is enough for comparing true vs spatial-null
    weighted_strength = float(np.average(
        df["mean_joint_score_active"].values,
        weights=np.maximum(df["n_active_edges"].values, 1)
    )) if len(df) > 0 else 0.0

    return {
        "baseline_interface_fraction": baseline_interface_fraction,
        "weighted_axis_strength": weighted_strength,
        "interface_proxy_score": baseline_interface_fraction * weighted_strength,
    }


# ---------------------------
# Train one run
# ---------------------------
def train_one_run(world_name: str, seed: int, verbose: bool = False):
    set_seed(seed)

    data_obj = dataset_artifacts["world_data"][world_name]
    data_dev = move_data_to_device(data_obj, DEVICE)

    model = make_model_for_world(world_name, data_obj)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )

    split_masks = dataset_artifacts["split_masks_by_seed"][seed]
    train_mask = torch.tensor(split_masks["train_mask"], dtype=torch.bool, device=DEVICE)
    val_mask = torch.tensor(split_masks["val_mask"], dtype=torch.bool, device=DEVICE)
    test_mask = torch.tensor(split_masks["test_mask"], dtype=torch.bool, device=DEVICE)

    history_rows = []
    best_state = None
    best_val_f1 = -np.inf

    for epoch in range(1, CONFIG["epochs"] + 1):
        # train
        model.train()
        optimizer.zero_grad()

        out = model(data_dev)
        logits = out["logits"]

        loss = F.cross_entropy(logits[train_mask], data_dev.y[train_mask])
        loss.backward()
        optimizer.step()

        # eval
        model.eval()
        with torch.no_grad():
            out_eval = model(data_dev)
            logits_eval = out_eval["logits"]
            prob_eval = F.softmax(logits_eval, dim=1)[:, 1]

            metrics_row = {"epoch": epoch, "train_loss": float(loss.item())}

            for split_name, mask in [("train", train_mask), ("val", val_mask), ("test", test_mask)]:
                y_true = tensor_to_numpy(data_dev.y[mask])
                y_prob = tensor_to_numpy(prob_eval[mask])
                y_pred = (y_prob >= 0.5).astype(int)

                m = compute_classification_metrics(y_true, y_prob, y_pred)
                metrics_row[f"{split_name}_accuracy"] = m["accuracy"]
                metrics_row[f"{split_name}_f1"] = m["f1"]
                metrics_row[f"{split_name}_roc_auc"] = m["roc_auc"]

            history_rows.append(metrics_row)

            if metrics_row["val_f1"] > best_val_f1:
                best_val_f1 = metrics_row["val_f1"]
                best_state = copy.deepcopy(model.state_dict())

        if verbose and (epoch == 1 or epoch % 10 == 0 or epoch == CONFIG["epochs"]):
            print(
                f"[{world_name} | seed={seed}] "
                f"epoch={epoch:03d} "
                f"loss={metrics_row['train_loss']:.4f} "
                f"val_f1={metrics_row['val_f1']:.4f} "
                f"test_f1={metrics_row['test_f1']:.4f}"
            )

    # restore best model
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        final_out = model(data_dev)
        final_logits = final_out["logits"]
        final_prob = F.softmax(final_logits, dim=1)[:, 1]
        final_pred = (final_prob >= 0.5).long()

        embedding = tensor_to_numpy(final_out["embedding"])
        gene_logits = tensor_to_numpy(final_out["gene_logits"])
        pathway_scores = tensor_to_numpy(final_out["pathway_scores"])

    history_df = pd.DataFrame(history_rows)

    # choose edge table by world
    if world_name in ["true", "mask_shuffle"]:
        edge_table_world = edge_artifacts["edge_table_true"].copy()
    elif world_name == "spatial_null":
        edge_table_world = edge_artifacts["edge_table_spatial_null"].copy()
    else:
        raise ValueError(f"Unknown world: {world_name}")

    ranked_axes = rank_axes_from_edge_table(edge_table_world)
    seeded_rank_summary = summarize_seeded_axis_ranks(ranked_axes)
    seed_vs_decoy = compute_seed_vs_decoy_margin(ranked_axes)

    pathway_alignment_df = compute_pathway_alignment(
        pathway_scores=pathway_scores,
        y_true_binary=dataset_artifacts["y"],
        pathway_names=pathway_artifacts["pathway_names"],
    )
    pathway_alignment_summary = summarize_pathway_alignment(pathway_alignment_df)

    interface_summary = compute_interface_edge_score(
        edge_index_tensor=data_obj.edge_index,
        edge_table_world=edge_table_world,
        region_id_array=dataset_artifacts["region_id"],
        axis_subset=[f"{l}->{r}" for l, r in seeded_axes],
    )

    run_summary = {
        "world": world_name,
        "seed": seed,
        "best_val_f1": float(history_df["val_f1"].max()),
        "best_test_f1": float(history_df.loc[history_df["val_f1"].idxmax(), "test_f1"]),
        "final_test_f1": float(history_df.iloc[-1]["test_f1"]),
        "final_test_auc": float(history_df.iloc[-1]["test_roc_auc"]) if pd.notnull(history_df.iloc[-1]["test_roc_auc"]) else np.nan,
        **pathway_alignment_summary,
        **seed_vs_decoy,
        **interface_summary,
    }

    return {
        "model_state_dict": model.state_dict(),
        "history_df": history_df,
        "embedding": embedding,
        "gene_logits": gene_logits,
        "pathway_scores": pathway_scores,
        "prob_positive": tensor_to_numpy(final_prob),
        "pred_label": tensor_to_numpy(final_pred),
        "ranked_axes_df": ranked_axes,
        "seeded_rank_summary_df": seeded_rank_summary,
        "pathway_alignment_df": pathway_alignment_df,
        "run_summary": pd.DataFrame([run_summary]),
    }


print("Training and evaluation helpers defined.")
print("Next cell will run the TRUE world across seeds.")

Training and evaluation helpers defined.
Next cell will run the TRUE world across seeds.


### Step 10

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [13]:
# CELL 11 — run the TRUE world across all seeds and save per-seed outputs

true_world_results = {}
true_world_run_summaries = []
true_world_seeded_rank_summaries = []
true_world_pathway_alignment = []
true_world_history_last = []

for seed in CONFIG["seeds"]:
    print(f"Running TRUE world | seed={seed}")
    result = train_one_run(world_name="true", seed=seed, verbose=True)
    true_world_results[seed] = result

    run_summary_df = result["run_summary"].copy()
    run_summary_df["world"] = "true"
    run_summary_df["seed"] = seed
    true_world_run_summaries.append(run_summary_df)

    seeded_rank_df = result["seeded_rank_summary_df"].copy()
    seeded_rank_df["world"] = "true"
    seeded_rank_df["seed"] = seed
    true_world_seeded_rank_summaries.append(seeded_rank_df)

    pathway_align_df = result["pathway_alignment_df"].copy()
    pathway_align_df["world"] = "true"
    pathway_align_df["seed"] = seed
    true_world_pathway_alignment.append(pathway_align_df)

    history_df = result["history_df"].copy()
    history_df["world"] = "true"
    history_df["seed"] = seed
    history_df["is_best_val_epoch"] = False
    best_idx = history_df["val_f1"].idxmax()
    history_df.loc[best_idx, "is_best_val_epoch"] = True
    true_world_history_last.append(history_df)

    if CONFIG["save_tables"]:
        result["history_df"].to_csv(TABLES / f"history_true_seed{seed}.csv", index=False)
        result["ranked_axes_df"].to_csv(TABLES / f"ranked_axes_true_seed{seed}.csv", index=False)
        result["seeded_rank_summary_df"].to_csv(TABLES / f"seeded_rank_summary_true_seed{seed}.csv", index=False)
        result["pathway_alignment_df"].to_csv(TABLES / f"pathway_alignment_true_seed{seed}.csv", index=False)

true_world_run_summary = pd.concat(true_world_run_summaries, axis=0, ignore_index=True)
true_world_seeded_ranks = pd.concat(true_world_seeded_rank_summaries, axis=0, ignore_index=True)
true_world_pathway_alignment_df = pd.concat(true_world_pathway_alignment, axis=0, ignore_index=True)
true_world_history_df = pd.concat(true_world_history_last, axis=0, ignore_index=True)

if CONFIG["save_tables"]:
    true_world_run_summary.to_csv(TABLES / "results_true_allseeds.csv", index=False)
    true_world_seeded_ranks.to_csv(TABLES / "seeded_ranks_true_allseeds.csv", index=False)
    true_world_pathway_alignment_df.to_csv(TABLES / "pathway_alignment_true_allseeds.csv", index=False)
    true_world_history_df.to_csv(TABLES / "history_true_allseeds.csv", index=False)

print("\nTRUE world complete.")
print("\nRun summary:")
display(true_world_run_summary)

print("\nSeeded axis ranks across seeds:")
display(true_world_seeded_ranks)

print("\nPathway alignment across seeds:")
display(true_world_pathway_alignment_df)

Running TRUE world | seed=11
[true | seed=11] epoch=001 loss=0.6899 val_f1=0.0000 test_f1=0.0000
[true | seed=11] epoch=010 loss=0.6725 val_f1=0.0000 test_f1=0.0000
[true | seed=11] epoch=020 loss=0.6308 val_f1=0.5566 test_f1=0.6175
[true | seed=11] epoch=030 loss=0.5559 val_f1=0.6207 test_f1=0.6222
[true | seed=11] epoch=040 loss=0.4878 val_f1=0.5877 test_f1=0.6038
[true | seed=11] epoch=050 loss=0.4705 val_f1=0.6026 test_f1=0.6175
Running TRUE world | seed=17
[true | seed=17] epoch=001 loss=0.6894 val_f1=0.0000 test_f1=0.0000
[true | seed=17] epoch=010 loss=0.6749 val_f1=0.0000 test_f1=0.0000
[true | seed=17] epoch=020 loss=0.6587 val_f1=0.0000 test_f1=0.0000
[true | seed=17] epoch=030 loss=0.6173 val_f1=0.0000 test_f1=0.0000
[true | seed=17] epoch=040 loss=0.5196 val_f1=0.6723 test_f1=0.6824
[true | seed=17] epoch=050 loss=0.4834 val_f1=0.6787 test_f1=0.6723
Running TRUE world | seed=23
[true | seed=23] epoch=001 loss=0.6816 val_f1=0.0000 test_f1=0.0000
[true | seed=23] epoch=010 lo

,world,seed,best_val_f1,best_test_f1,final_test_f1,final_test_auc,pathway_sign_match_rate,pathway_top4_overlap,seeded_mean_joint_score,decoy_mean_joint_score,seed_minus_decoy_margin,baseline_interface_fraction,weighted_axis_strength,interface_proxy_score
0,true,11,0.641975,0.622807,0.617512,0.796862,0.50,4,36.627422,30.978361,5.649061,0.191316,37.315454,7.139035
1,true,17,0.687783,0.666667,0.672269,0.810824,0.25,4,36.627422,30.978361,5.649061,0.191316,37.315454,7.139035
2,true,23,0.648148,0.671937,0.614035,0.768476,0.75,4,36.627422,30.978361,5.649061,0.191316,37.315454,7.139035



Seeded axis ranks across seeds:


,axis,axis_type,n_active_edges,active_edge_fraction,mean_joint_score_active,rank_joint_score,world,seed
0,TGFB1->TGFBR2,seeded,1259,0.082829,41.118393,1,true,11
1,CXCL12->CXCR4,seeded,730,0.048026,40.154732,2,true,11
2,IFNG->IFNGR1,seeded,788,0.051842,28.609142,16,true,11
3,TGFB1->TGFBR2,seeded,1259,0.082829,41.118393,1,true,17
4,CXCL12->CXCR4,seeded,730,0.048026,40.154732,2,true,17
5,IFNG->IFNGR1,seeded,788,0.051842,28.609142,16,true,17
6,TGFB1->TGFBR2,seeded,1259,0.082829,41.118393,1,true,23
7,CXCL12->CXCR4,seeded,730,0.048026,40.154732,2,true,23
8,IFNG->IFNGR1,seeded,788,0.051842,28.609142,16,true,23



Pathway alignment across seeds:


,pathway,mean_resistant_like,mean_other,delta_resistant_minus_other,expected_direction,observed_direction,sign_match,abs_delta,world,seed
0,TGFB,1.414504,0.344002,1.070502,up,up,True,1.070502,true,11
1,WNT,0.033149,0.190167,-0.157018,up,down,False,0.157018,true,11
2,IFNG,1.271544,0.308156,0.963387,down,up,False,0.963387,true,11
3,Antigen_Presentation,0.054175,0.308921,-0.254746,down,down,True,0.254746,true,11
4,TGFB,0.376413,2.141183,-1.764770,up,down,False,1.764770,true,17
5,WNT,1.411010,0.288418,1.122592,up,up,True,1.122592,true,17
6,IFNG,1.179569,0.244064,0.935505,down,up,False,0.935505,true,17
7,Antigen_Presentation,1.565094,0.330316,1.234778,down,up,False,1.234778,true,17
8,TGFB,0.402162,0.160191,0.241971,up,up,True,0.241971,true,23
9,WNT,0.719690,0.285582,0.434108,up,up,True,0.434108,true,23


#### Output interpretation
- Compare results to falsification expectations
- Look for separation between TRUE vs NULL conditions
- Strong separation indicates meaningful signal usage


### Step 11

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [14]:
# CELL 12 — run the MASK-SHUFFLE world across all seeds and save per-seed outputs

mask_shuffle_results = {}
mask_shuffle_run_summaries = []
mask_shuffle_seeded_rank_summaries = []
mask_shuffle_pathway_alignment = []
mask_shuffle_history_all = []

for seed in CONFIG["seeds"]:
    print(f"Running MASK-SHUFFLE world | seed={seed}")
    result = train_one_run(world_name="mask_shuffle", seed=seed, verbose=True)
    mask_shuffle_results[seed] = result

    run_summary_df = result["run_summary"].copy()
    run_summary_df["world"] = "mask_shuffle"
    run_summary_df["seed"] = seed
    mask_shuffle_run_summaries.append(run_summary_df)

    seeded_rank_df = result["seeded_rank_summary_df"].copy()
    seeded_rank_df["world"] = "mask_shuffle"
    seeded_rank_df["seed"] = seed
    mask_shuffle_seeded_rank_summaries.append(seeded_rank_df)

    pathway_align_df = result["pathway_alignment_df"].copy()
    pathway_align_df["world"] = "mask_shuffle"
    pathway_align_df["seed"] = seed
    mask_shuffle_pathway_alignment.append(pathway_align_df)

    history_df = result["history_df"].copy()
    history_df["world"] = "mask_shuffle"
    history_df["seed"] = seed
    history_df["is_best_val_epoch"] = False
    best_idx = history_df["val_f1"].idxmax()
    history_df.loc[best_idx, "is_best_val_epoch"] = True
    mask_shuffle_history_all.append(history_df)

    if CONFIG["save_tables"]:
        result["history_df"].to_csv(TABLES / f"history_maskshuffle_seed{seed}.csv", index=False)
        result["ranked_axes_df"].to_csv(TABLES / f"ranked_axes_maskshuffle_seed{seed}.csv", index=False)
        result["seeded_rank_summary_df"].to_csv(TABLES / f"seeded_rank_summary_maskshuffle_seed{seed}.csv", index=False)
        result["pathway_alignment_df"].to_csv(TABLES / f"pathway_alignment_maskshuffle_seed{seed}.csv", index=False)

mask_shuffle_run_summary = pd.concat(mask_shuffle_run_summaries, axis=0, ignore_index=True)
mask_shuffle_seeded_ranks = pd.concat(mask_shuffle_seeded_rank_summaries, axis=0, ignore_index=True)
mask_shuffle_pathway_alignment_df = pd.concat(mask_shuffle_pathway_alignment, axis=0, ignore_index=True)
mask_shuffle_history_df = pd.concat(mask_shuffle_history_all, axis=0, ignore_index=True)

if CONFIG["save_tables"]:
    mask_shuffle_run_summary.to_csv(TABLES / "results_maskshuffle_allseeds.csv", index=False)
    mask_shuffle_seeded_ranks.to_csv(TABLES / "seeded_ranks_maskshuffle_allseeds.csv", index=False)
    mask_shuffle_pathway_alignment_df.to_csv(TABLES / "pathway_alignment_maskshuffle_allseeds.csv", index=False)
    mask_shuffle_history_df.to_csv(TABLES / "history_maskshuffle_allseeds.csv", index=False)

print("\nMASK-SHUFFLE world complete.")
print("\nRun summary:")
display(mask_shuffle_run_summary)

print("\nSeeded axis ranks across seeds:")
display(mask_shuffle_seeded_ranks)

print("\nPathway alignment across seeds:")
display(mask_shuffle_pathway_alignment_df)

Running MASK-SHUFFLE world | seed=11
[mask_shuffle | seed=11] epoch=001 loss=0.6901 val_f1=0.0000 test_f1=0.0000
[mask_shuffle | seed=11] epoch=010 loss=0.6723 val_f1=0.0000 test_f1=0.0000
[mask_shuffle | seed=11] epoch=020 loss=0.6297 val_f1=0.5728 test_f1=0.6161
[mask_shuffle | seed=11] epoch=030 loss=0.5623 val_f1=0.6009 test_f1=0.6087
[mask_shuffle | seed=11] epoch=040 loss=0.5018 val_f1=0.6298 test_f1=0.6262
[mask_shuffle | seed=11] epoch=050 loss=0.4716 val_f1=0.5727 test_f1=0.5894
Running MASK-SHUFFLE world | seed=17
[mask_shuffle | seed=17] epoch=001 loss=0.6893 val_f1=0.0000 test_f1=0.0000
[mask_shuffle | seed=17] epoch=010 loss=0.6747 val_f1=0.0000 test_f1=0.0000
[mask_shuffle | seed=17] epoch=020 loss=0.6503 val_f1=0.0000 test_f1=0.0000
[mask_shuffle | seed=17] epoch=030 loss=0.5842 val_f1=0.0000 test_f1=0.0000
[mask_shuffle | seed=17] epoch=040 loss=0.5300 val_f1=0.6751 test_f1=0.6980
[mask_shuffle | seed=17] epoch=050 loss=0.4776 val_f1=0.6667 test_f1=0.6781
Running MASK-S

,world,seed,best_val_f1,best_test_f1,final_test_f1,final_test_auc,pathway_sign_match_rate,pathway_top4_overlap,seeded_mean_joint_score,decoy_mean_joint_score,seed_minus_decoy_margin,baseline_interface_fraction,weighted_axis_strength,interface_proxy_score
0,mask_shuffle,11,0.650000,0.651786,0.589372,0.794436,0.75,4,36.627422,30.978361,5.649061,0.191316,37.315454,7.139035
1,mask_shuffle,17,0.681223,0.693548,0.678112,0.814201,0.25,4,36.627422,30.978361,5.649061,0.191316,37.315454,7.139035
2,mask_shuffle,23,0.638889,0.640625,0.581818,0.756167,0.75,4,36.627422,30.978361,5.649061,0.191316,37.315454,7.139035



Seeded axis ranks across seeds:


,axis,axis_type,n_active_edges,active_edge_fraction,mean_joint_score_active,rank_joint_score,world,seed
0,TGFB1->TGFBR2,seeded,1259,0.082829,41.118393,1,mask_shuffle,11
1,CXCL12->CXCR4,seeded,730,0.048026,40.154732,2,mask_shuffle,11
2,IFNG->IFNGR1,seeded,788,0.051842,28.609142,16,mask_shuffle,11
3,TGFB1->TGFBR2,seeded,1259,0.082829,41.118393,1,mask_shuffle,17
4,CXCL12->CXCR4,seeded,730,0.048026,40.154732,2,mask_shuffle,17
5,IFNG->IFNGR1,seeded,788,0.051842,28.609142,16,mask_shuffle,17
6,TGFB1->TGFBR2,seeded,1259,0.082829,41.118393,1,mask_shuffle,23
7,CXCL12->CXCR4,seeded,730,0.048026,40.154732,2,mask_shuffle,23
8,IFNG->IFNGR1,seeded,788,0.051842,28.609142,16,mask_shuffle,23



Pathway alignment across seeds:


,pathway,mean_resistant_like,mean_other,delta_resistant_minus_other,expected_direction,observed_direction,sign_match,abs_delta,world,seed
0,TGFB,1.048462,0.250624,0.797838,up,up,True,0.797838,mask_shuffle,11
1,WNT,0.266723,0.102041,0.164683,up,up,True,0.164683,mask_shuffle,11
2,IFNG,1.498917,0.354029,1.144888,down,up,False,1.144888,mask_shuffle,11
3,Antigen_Presentation,0.056615,0.309110,-0.252495,down,down,True,0.252495,mask_shuffle,11
4,TGFB,0.334939,2.166154,-1.831215,up,down,False,1.831215,mask_shuffle,17
5,WNT,0.755678,0.216410,0.539267,up,up,True,0.539267,mask_shuffle,17
6,IFNG,0.536232,0.157161,0.379071,down,up,False,0.379071,mask_shuffle,17
7,Antigen_Presentation,0.747196,0.225681,0.521515,down,up,False,0.521515,mask_shuffle,17
8,TGFB,0.697345,0.320183,0.377162,up,up,True,0.377162,mask_shuffle,23
9,WNT,0.512770,0.235964,0.276806,up,up,True,0.276806,mask_shuffle,23


### Step 12

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [15]:
# CELL 13 — run the SPATIAL-NULL world across all seeds and save per-seed outputs

spatial_null_results = {}
spatial_null_run_summaries = []
spatial_null_seeded_rank_summaries = []
spatial_null_pathway_alignment = []
spatial_null_history_all = []

for seed in CONFIG["seeds"]:
    print(f"Running SPATIAL-NULL world | seed={seed}")
    result = train_one_run(world_name="spatial_null", seed=seed, verbose=True)
    spatial_null_results[seed] = result

    run_summary_df = result["run_summary"].copy()
    run_summary_df["world"] = "spatial_null"
    run_summary_df["seed"] = seed
    spatial_null_run_summaries.append(run_summary_df)

    seeded_rank_df = result["seeded_rank_summary_df"].copy()
    seeded_rank_df["world"] = "spatial_null"
    seeded_rank_df["seed"] = seed
    spatial_null_seeded_rank_summaries.append(seeded_rank_df)

    pathway_align_df = result["pathway_alignment_df"].copy()
    pathway_align_df["world"] = "spatial_null"
    spatial_null_pathway_alignment.append(pathway_align_df.assign(seed=seed))

    history_df = result["history_df"].copy()
    history_df["world"] = "spatial_null"
    history_df["seed"] = seed
    history_df["is_best_val_epoch"] = False
    best_idx = history_df["val_f1"].idxmax()
    history_df.loc[best_idx, "is_best_val_epoch"] = True
    spatial_null_history_all.append(history_df)

    if CONFIG["save_tables"]:
        result["history_df"].to_csv(TABLES / f"history_spatialnull_seed{seed}.csv", index=False)
        result["ranked_axes_df"].to_csv(TABLES / f"ranked_axes_spatialnull_seed{seed}.csv", index=False)
        result["seeded_rank_summary_df"].to_csv(TABLES / f"seeded_rank_summary_spatialnull_seed{seed}.csv", index=False)
        result["pathway_alignment_df"].to_csv(TABLES / f"pathway_alignment_spatialnull_seed{seed}.csv", index=False)

spatial_null_run_summary = pd.concat(spatial_null_run_summaries, axis=0, ignore_index=True)
spatial_null_seeded_ranks = pd.concat(spatial_null_seeded_rank_summaries, axis=0, ignore_index=True)
spatial_null_pathway_alignment_df = pd.concat(spatial_null_pathway_alignment, axis=0, ignore_index=True)
spatial_null_history_df = pd.concat(spatial_null_history_all, axis=0, ignore_index=True)

if CONFIG["save_tables"]:
    spatial_null_run_summary.to_csv(TABLES / "results_spatialnull_allseeds.csv", index=False)
    spatial_null_seeded_ranks.to_csv(TABLES / "seeded_ranks_spatialnull_allseeds.csv", index=False)
    spatial_null_pathway_alignment_df.to_csv(TABLES / "pathway_alignment_spatialnull_allseeds.csv", index=False)
    spatial_null_history_df.to_csv(TABLES / "history_spatialnull_allseeds.csv", index=False)

print("\nSPATIAL-NULL world complete.")
print("\nRun summary:")
display(spatial_null_run_summary)

print("\nSeeded axis ranks across seeds:")
display(spatial_null_seeded_ranks)

print("\nPathway alignment across seeds:")
display(spatial_null_pathway_alignment_df)

Running SPATIAL-NULL world | seed=11
[spatial_null | seed=11] epoch=001 loss=0.6898 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=11] epoch=010 loss=0.6779 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=11] epoch=020 loss=0.6691 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=11] epoch=030 loss=0.6691 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=11] epoch=040 loss=0.6636 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=11] epoch=050 loss=0.6180 val_f1=0.0000 test_f1=0.0000
Running SPATIAL-NULL world | seed=17
[spatial_null | seed=17] epoch=001 loss=0.6893 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=17] epoch=010 loss=0.6784 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=17] epoch=020 loss=0.6784 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=17] epoch=030 loss=0.6764 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=17] epoch=040 loss=0.6757 val_f1=0.0000 test_f1=0.0000
[spatial_null | seed=17] epoch=050 loss=0.6754 val_f1=0.0000 test_f1=0.0000
Running SPATIA

,world,seed,best_val_f1,best_test_f1,final_test_f1,final_test_auc,pathway_sign_match_rate,pathway_top4_overlap,seeded_mean_joint_score,decoy_mean_joint_score,seed_minus_decoy_margin,baseline_interface_fraction,weighted_axis_strength,interface_proxy_score
0,spatial_null,11,0.000000,0.000000,0.000000,0.715886,0.25,4,36.606785,30.95546,5.651326,0.291382,37.079734,10.804351
1,spatial_null,17,0.000000,0.000000,0.000000,0.606380,0.50,4,36.606785,30.95546,5.651326,0.291382,37.079734,10.804351
2,spatial_null,23,0.515556,0.561538,0.553846,0.607117,0.50,4,36.606785,30.95546,5.651326,0.291382,37.079734,10.804351



Seeded axis ranks across seeds:


,axis,axis_type,n_active_edges,active_edge_fraction,mean_joint_score_active,rank_joint_score,world,seed
0,TGFB1->TGFBR2,seeded,942,0.061974,40.737885,1,spatial_null,11
1,CXCL12->CXCR4,seeded,909,0.059803,40.473831,2,spatial_null,11
2,IFNG->IFNGR1,seeded,771,0.050724,28.608641,16,spatial_null,11
3,TGFB1->TGFBR2,seeded,942,0.061974,40.737885,1,spatial_null,17
4,CXCL12->CXCR4,seeded,909,0.059803,40.473831,2,spatial_null,17
5,IFNG->IFNGR1,seeded,771,0.050724,28.608641,16,spatial_null,17
6,TGFB1->TGFBR2,seeded,942,0.061974,40.737885,1,spatial_null,23
7,CXCL12->CXCR4,seeded,909,0.059803,40.473831,2,spatial_null,23
8,IFNG->IFNGR1,seeded,771,0.050724,28.608641,16,spatial_null,23



Pathway alignment across seeds:


,pathway,mean_resistant_like,mean_other,delta_resistant_minus_other,expected_direction,observed_direction,sign_match,abs_delta,world,seed
0,TGFB,0.000055,0.000076,-0.000021,up,down,False,0.000021,spatial_null,11
1,WNT,0.001257,0.001454,-0.000197,up,down,False,0.000197,spatial_null,11
2,IFNG,0.000330,0.000356,-0.000026,down,down,True,0.000026,spatial_null,11
3,Antigen_Presentation,0.004373,0.004344,0.000029,down,up,False,0.000029,spatial_null,11
4,TGFB,0.000073,0.000077,-0.000005,up,down,False,0.000005,spatial_null,17
5,WNT,0.001676,0.001617,0.000059,up,up,True,0.000059,spatial_null,17
6,IFNG,0.004214,0.004187,0.000027,down,up,False,0.000027,spatial_null,17
7,Antigen_Presentation,0.004517,0.004589,-0.000071,down,down,True,0.000071,spatial_null,17
8,TGFB,0.042273,0.017778,0.024495,up,up,True,0.024495,spatial_null,23
9,WNT,0.926295,0.407458,0.518837,up,up,True,0.518837,spatial_null,23


### Step 13

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [16]:
# CELL 14 — aggregate the valid world-level results and compare true vs null worlds
# Valid at this stage:
# - classification performance
# - pathway alignment
#
# Not yet valid for falsification:
# - seeded rank
# - seed-vs-decoy margin
# - interface proxy
# because those are currently derived from static edge tables, not model outputs.

all_run_summaries = pd.concat(
    [
        true_world_run_summary,
        mask_shuffle_run_summary,
        spatial_null_run_summary,
    ],
    axis=0,
    ignore_index=True,
)

# Aggregate only the metrics that are already valid
valid_metric_cols = [
    "best_val_f1",
    "best_test_f1",
    "final_test_f1",
    "final_test_auc",
    "pathway_sign_match_rate",
    "pathway_top4_overlap",
]

world_valid_summary = (
    all_run_summaries
    .groupby("world")[valid_metric_cols]
    .agg(["mean", "std", "min", "max"])
)

# Flatten columns
world_valid_summary.columns = [
    f"{metric}_{stat}" for metric, stat in world_valid_summary.columns
]
world_valid_summary = world_valid_summary.reset_index()

# Seed-level compact view
seed_level_valid = all_run_summaries[
    ["world", "seed"] + valid_metric_cols
].copy().sort_values(["world", "seed"]).reset_index(drop=True)

# Simple pairwise comparisons vs true world
def compare_worlds(metric: str, baseline: str = "true", other_worlds=("mask_shuffle", "spatial_null")):
    baseline_vals = all_run_summaries.loc[all_run_summaries["world"] == baseline, metric].values
    rows = []
    for w in other_worlds:
        other_vals = all_run_summaries.loc[all_run_summaries["world"] == w, metric].values
        rows.append({
            "metric": metric,
            "baseline_world": baseline,
            "comparison_world": w,
            "baseline_mean": float(np.mean(baseline_vals)),
            "comparison_mean": float(np.mean(other_vals)),
            "delta_baseline_minus_comparison": float(np.mean(baseline_vals) - np.mean(other_vals)),
        })
    return pd.DataFrame(rows)

comparison_tables = []
for metric in valid_metric_cols:
    comparison_tables.append(compare_worlds(metric))

world_comparisons = pd.concat(comparison_tables, axis=0, ignore_index=True)

if CONFIG["save_tables"]:
    all_run_summaries.to_csv(TABLES / "all_run_summaries_raw.csv", index=False)
    world_valid_summary.to_csv(TABLES / "world_valid_summary.csv", index=False)
    seed_level_valid.to_csv(TABLES / "seed_level_valid_metrics.csv", index=False)
    world_comparisons.to_csv(TABLES / "world_valid_metric_comparisons.csv", index=False)

print("Aggregated valid world-level summaries.")
print("\nWorld-level summary:")
display(world_valid_summary)

print("\nSeed-level valid metrics:")
display(seed_level_valid)

print("\nComparisons vs TRUE world:")
display(world_comparisons)

Aggregated valid world-level summaries.

World-level summary:


,world,best_val_f1_mean,best_val_f1_std,best_val_f1_min,best_val_f1_max,best_test_f1_mean,best_test_f1_std,best_test_f1_min,best_test_f1_max,final_test_f1_mean,...,final_test_auc_min,final_test_auc_max,pathway_sign_match_rate_mean,pathway_sign_match_rate_std,pathway_sign_match_rate_min,pathway_sign_match_rate_max,pathway_top4_overlap_mean,pathway_top4_overlap_std,pathway_top4_overlap_min,pathway_top4_overlap_max
0,mask_shuffle,0.656704,0.021949,0.638889,0.681223,0.661986,0.027897,0.640625,0.693548,0.616434,...,0.756167,0.814201,0.583333,0.288675,0.25,0.75,4.0,0.0,4,4
1,spatial_null,0.171852,0.297656,0.000000,0.515556,0.187179,0.324204,0.000000,0.561538,0.184615,...,0.606380,0.715886,0.416667,0.144338,0.25,0.50,4.0,0.0,4,4
2,true,0.659302,0.024857,0.641975,0.687783,0.653803,0.026973,0.622807,0.671937,0.634605,...,0.768476,0.810824,0.500000,0.250000,0.25,0.75,4.0,0.0,4,4



Seed-level valid metrics:


,world,seed,best_val_f1,best_test_f1,final_test_f1,final_test_auc,pathway_sign_match_rate,pathway_top4_overlap
0,mask_shuffle,11,0.650000,0.651786,0.589372,0.794436,0.75,4
1,mask_shuffle,17,0.681223,0.693548,0.678112,0.814201,0.25,4
2,mask_shuffle,23,0.638889,0.640625,0.581818,0.756167,0.75,4
3,spatial_null,11,0.000000,0.000000,0.000000,0.715886,0.25,4
4,spatial_null,17,0.000000,0.000000,0.000000,0.606380,0.50,4
5,spatial_null,23,0.515556,0.561538,0.553846,0.607117,0.50,4
6,true,11,0.641975,0.622807,0.617512,0.796862,0.50,4
7,true,17,0.687783,0.666667,0.672269,0.810824,0.25,4
8,true,23,0.648148,0.671937,0.614035,0.768476,0.75,4



Comparisons vs TRUE world:


,metric,baseline_world,comparison_world,baseline_mean,comparison_mean,delta_baseline_minus_comparison
0,best_val_f1,true,mask_shuffle,0.659302,0.656704,0.002598
1,best_val_f1,true,spatial_null,0.659302,0.171852,0.487450
2,best_test_f1,true,mask_shuffle,0.653803,0.661986,-0.008183
3,best_test_f1,true,spatial_null,0.653803,0.187179,0.466624
4,final_test_f1,true,mask_shuffle,0.634605,0.616434,0.018171
5,final_test_f1,true,spatial_null,0.634605,0.184615,0.449990
6,final_test_auc,true,mask_shuffle,0.792054,0.788268,0.003786
7,final_test_auc,true,spatial_null,0.792054,0.643127,0.148927
8,pathway_sign_match_rate,true,mask_shuffle,0.500000,0.583333,-0.083333
9,pathway_sign_match_rate,true,spatial_null,0.500000,0.416667,0.083333


### Correct architecture

### Step 15

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [35]:
# CELL 18 — stricter corrected bottleneck model and training setup
# Replaces previous corrected model cell.

CONFIG["learning_rate_corrected"] = 5e-4
CONFIG["hidden_dim_corrected"] = 8
CONFIG["dropout_corrected"] = 0.10
CONFIG["lambda_mask_l1"] = 1e-5

DEFAULT_PATHWAYS = ["TGFB", "WNT", "IFNG", "Antigen_Presentation"]


class SpatialMMKPNN_StrictMask(nn.Module):

    def __init__(
        self,
        n_input_genes: int,
        n_pathways: int,
        hidden_dim: int,
        pathway_mask: torch.Tensor,
        dropout: float = 0.10,
    ):
        super().__init__()

        self.n_input_genes = n_input_genes
        self.n_pathways = n_pathways
        self.hidden_dim = hidden_dim
        self.dropout = dropout

        self.register_buffer("pathway_mask", pathway_mask.clone().float())

        self.gene_to_pathway_weight = nn.Parameter(torch.zeros(n_input_genes, n_pathways))
        self.pathway_bias = nn.Parameter(torch.zeros(n_pathways))

        self.pathway_encoder = nn.Linear(n_pathways, n_pathways * hidden_dim)

        self.pathway_gat = GATConv(
            in_channels=n_pathways * hidden_dim,
            out_channels=n_pathways * hidden_dim,
            heads=1,
            concat=False,
            dropout=dropout,
        )

        self.pre_gat_norm = nn.LayerNorm(n_pathways * hidden_dim)
        self.post_gat_norm = nn.LayerNorm(n_pathways * hidden_dim)

        self.channel_pool = nn.Linear(hidden_dim, 1)

        self.classifier = nn.Linear(n_pathways, 2)

        self.reset_parameters()

    def reset_parameters(self):
        with torch.no_grad():
            init = torch.randn_like(self.gene_to_pathway_weight) * 0.01 + 0.03
            self.gene_to_pathway_weight.copy_(init * self.pathway_mask)
            self.pathway_bias.zero_()

        self.pathway_encoder.reset_parameters()
        self.pathway_gat.reset_parameters()
        self.channel_pool.reset_parameters()
        self.classifier.reset_parameters()

    def forward(self, data: Data):

        x, edge_index = data.x, data.edge_index

        masked_weight = self.gene_to_pathway_weight * self.pathway_mask

        pathway_input = F.softplus(x @ masked_weight + self.pathway_bias)

        pathway_hidden = self.pathway_encoder(pathway_input)
        pathway_hidden = F.softplus(pathway_hidden)
        pathway_hidden = self.pre_gat_norm(pathway_hidden)

        pathway_hidden = self.pathway_gat(pathway_hidden, edge_index)
        pathway_hidden = F.softplus(pathway_hidden)
        pathway_hidden = self.post_gat_norm(pathway_hidden)

        pathway_hidden = F.dropout(pathway_hidden, p=self.dropout, training=self.training)

        pathway_hidden = pathway_hidden.view(-1, self.n_pathways, self.hidden_dim)

        pathway_scores = self.channel_pool(pathway_hidden).squeeze(-1)
        pathway_scores = F.softplus(pathway_scores)

        logits = self.classifier(pathway_scores)

        return {
            "logits": logits,
            "embedding": pathway_scores,
            "pathway_input": pathway_input,
            "pathway_scores": pathway_scores,
            "masked_weight": masked_weight,
        }


def make_stable_corrected_model_for_world(world_name: str, data_obj: Data):

    model = SpatialMMKPNN_StrictMask(
        n_input_genes=data_obj.x.shape[1],
        n_pathways=data_obj.pathway_mask.shape[1],
        hidden_dim=CONFIG["hidden_dim_corrected"],
        pathway_mask=data_obj.pathway_mask,
        dropout=CONFIG["dropout_corrected"],
    )

    return model.to(DEVICE)


def train_one_run_corrected_stable(world_name: str, seed: int, verbose: bool = False):

    set_seed(seed)

    data_obj = dataset_artifacts["world_data"][world_name]
    data_dev = move_data_to_device(data_obj, DEVICE)

    model = make_stable_corrected_model_for_world(world_name, data_obj)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG["learning_rate_corrected"],
        weight_decay=CONFIG["weight_decay"],
    )

    split_masks = dataset_artifacts["split_masks_by_seed"][seed]

    train_mask = torch.tensor(split_masks["train_mask"], dtype=torch.bool, device=DEVICE)
    val_mask = torch.tensor(split_masks["val_mask"], dtype=torch.bool, device=DEVICE)
    test_mask = torch.tensor(split_masks["test_mask"], dtype=torch.bool, device=DEVICE)

    y_train = data_dev.y[train_mask]

    n_pos = int((y_train == 1).sum().item())
    n_neg = int((y_train == 0).sum().item())

    pos_weight = max(n_neg / max(n_pos, 1), 1.0)

    class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float32, device=DEVICE)

    history_rows = []

    best_state = None
    best_val_f1 = -np.inf
    best_test_f1 = -np.inf

    for epoch in range(1, CONFIG["epochs"] + 1):

        model.train()
        optimizer.zero_grad()

        out = model(data_dev)

        logits = out["logits"]

        ce_loss = F.cross_entropy(
            logits[train_mask],
            data_dev.y[train_mask],
            weight=class_weights,
        )

        l1_loss = CONFIG["lambda_mask_l1"] * out["masked_weight"].abs().sum()

        loss = ce_loss + l1_loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        model.eval()

        with torch.no_grad():

            out_eval = model(data_dev)

            logits_eval = out_eval["logits"]

            prob_eval = F.softmax(logits_eval, dim=1)[:, 1]

            metrics_row = {
                "epoch": epoch,
                "train_loss": float(loss.item()),
            }

            for split_name, mask in [
                ("train", train_mask),
                ("val", val_mask),
                ("test", test_mask),
            ]:

                y_true = tensor_to_numpy(data_dev.y[mask])
                y_prob = tensor_to_numpy(prob_eval[mask])
                y_pred = (y_prob >= 0.5).astype(int)

                m = compute_classification_metrics(y_true, y_prob, y_pred)

                metrics_row[f"{split_name}_f1"] = m["f1"]

            history_rows.append(metrics_row)

            if metrics_row["val_f1"] > best_val_f1:

                best_val_f1 = metrics_row["val_f1"]
                best_test_f1 = metrics_row["test_f1"]

                best_state = copy.deepcopy(model.state_dict())

        if verbose and (epoch == 1 or epoch % 10 == 0 or epoch == CONFIG["epochs"]):

            print(
                f"[stable corrected {world_name} | seed={seed}] "
                f"epoch={epoch:03d} "
                f"loss={metrics_row['train_loss']:.4f} "
                f"val_f1={metrics_row['val_f1']:.4f} "
                f"test_f1={metrics_row['test_f1']:.4f}"
            )

    model.load_state_dict(best_state)

    model.eval()

    with torch.no_grad():

        final_out = model(data_dev)

        final_logits = final_out["logits"]

        final_prob = F.softmax(final_logits, dim=1)[:, 1]

    y_true_test = tensor_to_numpy(data_dev.y[test_mask])
    y_prob_test = tensor_to_numpy(final_prob[test_mask])
    y_pred_test = (y_prob_test >= 0.5).astype(int)

    test_metrics = compute_classification_metrics(y_true_test, y_prob_test, y_pred_test)

    pathway_scores = tensor_to_numpy(final_out["pathway_scores"])

    pathway_alignment_df = compute_pathway_alignment(
        pathway_scores=pathway_scores[test_mask.detach().cpu().numpy()],
        y_true_binary=tensor_to_numpy(data_dev.y[test_mask]),
        pathway_names=DEFAULT_PATHWAYS,
    )

    pathway_summary = summarize_pathway_alignment(pathway_alignment_df)

    run_summary = pd.DataFrame(
        [
            {
                "world": world_name,
                "seed": seed,
                "best_val_f1": float(best_val_f1),
                "best_test_f1": float(best_test_f1),
                "final_test_f1": float(test_metrics["f1"]),
                "final_test_auc": float(test_metrics["roc_auc"]),
                "pathway_sign_match_rate": pathway_summary["pathway_sign_match_rate"],
                "pathway_top4_overlap": pathway_summary["pathway_top4_overlap"],
            }
        ]
    )

    return {
        "world": world_name,
        "seed": seed,
        "run_summary": run_summary,
        "history_df": pd.DataFrame(history_rows),
        "pathway_alignment_df": pathway_alignment_df,
    }

#### Output interpretation
- Compare results to falsification expectations
- Look for separation between TRUE vs NULL conditions
- Strong separation indicates meaningful signal usage


### Step 16

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [36]:
# CELL 19 — run the stable corrected TRUE world across all seeds and save outputs

stable_corr_true_results = {}
stable_corr_true_run_summaries = []
stable_corr_true_pathway_alignment = []
stable_corr_true_history_all = []

for seed in CONFIG["seeds"]:
    print(f"Running stable corrected TRUE world | seed={seed}")
    result = train_one_run_corrected_stable(world_name="true", seed=seed, verbose=True)
    stable_corr_true_results[seed] = result

    run_summary_df = result["run_summary"].copy()
    run_summary_df["world"] = "true"
    run_summary_df["seed"] = seed
    stable_corr_true_run_summaries.append(run_summary_df)

    pathway_align_df = result["pathway_alignment_df"].copy()
    pathway_align_df["world"] = "true"
    pathway_align_df["seed"] = seed
    stable_corr_true_pathway_alignment.append(pathway_align_df)

    history_df = result["history_df"].copy()
    history_df["world"] = "true"
    history_df["seed"] = seed
    history_df["is_best_val_epoch"] = False
    best_idx = history_df["val_f1"].idxmax()
    history_df.loc[best_idx, "is_best_val_epoch"] = True
    stable_corr_true_history_all.append(history_df)

    if CONFIG["save_tables"]:
        result["history_df"].to_csv(TABLES / f"stable_corr_history_true_seed{seed}.csv", index=False)
        result["pathway_alignment_df"].to_csv(TABLES / f"stable_corr_pathway_alignment_true_seed{seed}.csv", index=False)

stable_corr_true_run_summary = pd.concat(stable_corr_true_run_summaries, axis=0, ignore_index=True)
stable_corr_true_pathway_alignment_df = pd.concat(stable_corr_true_pathway_alignment, axis=0, ignore_index=True)
stable_corr_true_history_df = pd.concat(stable_corr_true_history_all, axis=0, ignore_index=True)

if CONFIG["save_tables"]:
    stable_corr_true_run_summary.to_csv(TABLES / "stable_corr_results_true_allseeds.csv", index=False)
    stable_corr_true_pathway_alignment_df.to_csv(TABLES / "stable_corr_pathway_alignment_true_allseeds.csv", index=False)
    stable_corr_true_history_df.to_csv(TABLES / "stable_corr_history_true_allseeds.csv", index=False)

print("\nStable corrected TRUE world complete.")
print("\nRun summary:")
display(stable_corr_true_run_summary)

print("\nPathway alignment across seeds:")
display(stable_corr_true_pathway_alignment_df)

Running stable corrected TRUE world | seed=11
[stable corrected true | seed=11] epoch=001 loss=0.6987 val_f1=0.0000 test_f1=0.0000
[stable corrected true | seed=11] epoch=010 loss=0.6934 val_f1=0.6165 test_f1=0.5642
[stable corrected true | seed=11] epoch=020 loss=0.6888 val_f1=0.7089 test_f1=0.6810
[stable corrected true | seed=11] epoch=030 loss=0.6862 val_f1=0.3926 test_f1=0.5217
[stable corrected true | seed=11] epoch=040 loss=0.6768 val_f1=0.6578 test_f1=0.7032
[stable corrected true | seed=11] epoch=050 loss=0.6638 val_f1=0.6486 test_f1=0.6825
Running stable corrected TRUE world | seed=17
[stable corrected true | seed=17] epoch=001 loss=1.0147 val_f1=0.5195 test_f1=0.5961
[stable corrected true | seed=17] epoch=010 loss=0.8991 val_f1=0.5195 test_f1=0.5961
[stable corrected true | seed=17] epoch=020 loss=0.8190 val_f1=0.5195 test_f1=0.5961
[stable corrected true | seed=17] epoch=030 loss=0.7707 val_f1=0.5195 test_f1=0.5961
[stable corrected true | seed=17] epoch=040 loss=0.7378 va

,world,seed,best_val_f1,best_test_f1,final_test_f1,final_test_auc,pathway_sign_match_rate,pathway_top4_overlap
0,true,11,0.726688,0.695946,0.695946,0.855853,1.00,4
1,true,17,0.519481,0.596059,0.596059,0.849476,1.00,4
2,true,23,0.515625,0.564232,0.564232,0.239626,0.25,4



Pathway alignment across seeds:


,pathway,mean_resistant_like,mean_other,delta_resistant_minus_other,expected_direction,observed_direction,sign_match,abs_delta,world,seed
0,TGFB,0.975455,0.932171,0.043284,up,up,True,0.043284,true,11
1,WNT,1.605987,1.594220,0.011767,up,up,True,0.011767,true,11
2,IFNG,0.934607,0.952676,-0.018069,down,down,True,0.018069,true,11
3,Antigen_Presentation,0.634253,0.677830,-0.043577,down,down,True,0.043577,true,11
4,TGFB,0.757946,0.746074,0.011872,up,up,True,0.011872,true,17
5,WNT,1.276798,1.223193,0.053606,up,up,True,0.053606,true,17
6,IFNG,0.452703,0.456269,-0.003566,down,down,True,0.003566,true,17
7,Antigen_Presentation,0.712156,0.721565,-0.009409,down,down,True,0.009409,true,17
8,TGFB,0.932435,0.953938,-0.021503,up,down,False,0.021503,true,23
9,WNT,0.605888,0.599887,0.006001,up,up,True,0.006001,true,23


### Step 17

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [38]:
# CELL 20 — run the stable corrected MASK-SHUFFLE world across all seeds and save outputs

stable_corr_mask_results = {}
stable_corr_mask_run_summaries = []
stable_corr_mask_pathway_alignment = []
stable_corr_mask_history_all = []

for seed in CONFIG["seeds"]:
    print(f"Running stable corrected MASK-SHUFFLE world | seed={seed}")
    result = train_one_run_corrected_stable(world_name="mask_shuffle", seed=seed, verbose=True)
    stable_corr_mask_results[seed] = result

    run_summary_df = result["run_summary"].copy()
    run_summary_df["world"] = "mask_shuffle"
    run_summary_df["seed"] = seed
    stable_corr_mask_run_summaries.append(run_summary_df)

    pathway_align_df = result["pathway_alignment_df"].copy()
    pathway_align_df["world"] = "mask_shuffle"
    pathway_align_df["seed"] = seed
    stable_corr_mask_pathway_alignment.append(pathway_align_df)

    history_df = result["history_df"].copy()
    history_df["world"] = "mask_shuffle"
    history_df["seed"] = seed
    history_df["is_best_val_epoch"] = False
    best_idx = history_df["val_f1"].idxmax()
    history_df.loc[best_idx, "is_best_val_epoch"] = True
    stable_corr_mask_history_all.append(history_df)

    if CONFIG["save_tables"]:
        result["history_df"].to_csv(TABLES / f"stable_corr_history_maskshuffle_seed{seed}.csv", index=False)
        result["pathway_alignment_df"].to_csv(TABLES / f"stable_corr_pathway_alignment_maskshuffle_seed{seed}.csv", index=False)

stable_corr_mask_run_summary = pd.concat(stable_corr_mask_run_summaries, axis=0, ignore_index=True)
stable_corr_mask_pathway_alignment_df = pd.concat(stable_corr_mask_pathway_alignment, axis=0, ignore_index=True)
stable_corr_mask_history_df = pd.concat(stable_corr_mask_history_all, axis=0, ignore_index=True)

if CONFIG["save_tables"]:
    stable_corr_mask_run_summary.to_csv(TABLES / "stable_corr_results_maskshuffle_allseeds.csv", index=False)
    stable_corr_mask_pathway_alignment_df.to_csv(TABLES / "stable_corr_pathway_alignment_maskshuffle_allseeds.csv", index=False)
    stable_corr_mask_history_df.to_csv(TABLES / "stable_corr_history_maskshuffle_allseeds.csv", index=False)

print("\nStable corrected MASK-SHUFFLE world complete.")
print("\nRun summary:")
display(stable_corr_mask_run_summary)

print("\nPathway alignment across seeds:")
display(stable_corr_mask_pathway_alignment_df)

Running stable corrected MASK-SHUFFLE world | seed=11
[stable corrected mask_shuffle | seed=11] epoch=001 loss=0.7021 val_f1=0.0000 test_f1=0.0000
[stable corrected mask_shuffle | seed=11] epoch=010 loss=0.6974 val_f1=0.6165 test_f1=0.5642
[stable corrected mask_shuffle | seed=11] epoch=020 loss=0.6952 val_f1=0.6165 test_f1=0.5642
[stable corrected mask_shuffle | seed=11] epoch=030 loss=0.6958 val_f1=0.0000 test_f1=0.0522
[stable corrected mask_shuffle | seed=11] epoch=040 loss=0.6918 val_f1=0.7055 test_f1=0.6738
[stable corrected mask_shuffle | seed=11] epoch=050 loss=0.6891 val_f1=0.6442 test_f1=0.5978
Running stable corrected MASK-SHUFFLE world | seed=17
[stable corrected mask_shuffle | seed=17] epoch=001 loss=1.0235 val_f1=0.5195 test_f1=0.5961
[stable corrected mask_shuffle | seed=17] epoch=010 loss=0.9066 val_f1=0.5195 test_f1=0.5961
[stable corrected mask_shuffle | seed=17] epoch=020 loss=0.8255 val_f1=0.5195 test_f1=0.5961
[stable corrected mask_shuffle | seed=17] epoch=030 los

,world,seed,best_val_f1,best_test_f1,final_test_f1,final_test_auc,pathway_sign_match_rate,pathway_top4_overlap
0,mask_shuffle,11,0.705479,0.673835,0.673835,0.794798,1.00,4
1,mask_shuffle,17,0.519481,0.596059,0.596059,0.653346,1.00,4
2,mask_shuffle,23,0.515625,0.564232,0.564232,0.755496,0.75,4



Pathway alignment across seeds:


,pathway,mean_resistant_like,mean_other,delta_resistant_minus_other,expected_direction,observed_direction,sign_match,abs_delta,world,seed
0,TGFB,1.018567,1.004171,0.014396,up,up,True,0.014396,mask_shuffle,11
1,WNT,1.538290,1.529965,0.008325,up,up,True,0.008325,mask_shuffle,11
2,IFNG,0.966919,0.971462,-0.004542,down,down,True,0.004542,mask_shuffle,11
3,Antigen_Presentation,0.621499,0.632902,-0.011403,down,down,True,0.011403,mask_shuffle,11
4,TGFB,0.754467,0.748892,0.005574,up,up,True,0.005574,mask_shuffle,17
5,WNT,1.245301,1.243250,0.002051,up,up,True,0.002051,mask_shuffle,17
6,IFNG,0.453028,0.455382,-0.002354,down,down,True,0.002354,mask_shuffle,17
7,Antigen_Presentation,0.716915,0.717883,-0.000968,down,down,True,0.000968,mask_shuffle,17
8,TGFB,0.944462,0.944028,0.000433,up,up,True,0.000433,mask_shuffle,23
9,WNT,0.606591,0.600422,0.006169,up,up,True,0.006169,mask_shuffle,23


### Step 18

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


In [39]:
# CELL 21 — run the stable corrected SPATIAL-NULL world across all seeds and save outputs

stable_corr_null_results = {}
stable_corr_null_run_summaries = []
stable_corr_null_pathway_alignment = []
stable_corr_null_history_all = []

for seed in CONFIG["seeds"]:
    print(f"Running stable corrected SPATIAL-NULL world | seed={seed}")
    result = train_one_run_corrected_stable(world_name="spatial_null", seed=seed, verbose=True)
    stable_corr_null_results[seed] = result

    run_summary_df = result["run_summary"].copy()
    run_summary_df["world"] = "spatial_null"
    run_summary_df["seed"] = seed
    stable_corr_null_run_summaries.append(run_summary_df)

    pathway_align_df = result["pathway_alignment_df"].copy()
    pathway_align_df["world"] = "spatial_null"
    pathway_align_df["seed"] = seed
    stable_corr_null_pathway_alignment.append(pathway_align_df)

    history_df = result["history_df"].copy()
    history_df["world"] = "spatial_null"
    history_df["seed"] = seed
    history_df["is_best_val_epoch"] = False
    best_idx = history_df["val_f1"].idxmax()
    history_df.loc[best_idx, "is_best_val_epoch"] = True
    stable_corr_null_history_all.append(history_df)

    if CONFIG["save_tables"]:
        result["history_df"].to_csv(TABLES / f"stable_corr_history_spatialnull_seed{seed}.csv", index=False)
        result["pathway_alignment_df"].to_csv(TABLES / f"stable_corr_pathway_alignment_spatialnull_seed{seed}.csv", index=False)

stable_corr_null_run_summary = pd.concat(stable_corr_null_run_summaries, axis=0, ignore_index=True)
stable_corr_null_pathway_alignment_df = pd.concat(stable_corr_null_pathway_alignment, axis=0, ignore_index=True)
stable_corr_null_history_df = pd.concat(stable_corr_null_history_all, axis=0, ignore_index=True)

if CONFIG["save_tables"]:
    stable_corr_null_run_summary.to_csv(TABLES / "stable_corr_results_spatialnull_allseeds.csv", index=False)
    stable_corr_null_pathway_alignment_df.to_csv(TABLES / "stable_corr_pathway_alignment_spatialnull_allseeds.csv", index=False)
    stable_corr_null_history_df.to_csv(TABLES / "stable_corr_history_spatialnull_allseeds.csv", index=False)

print("\nStable corrected SPATIAL-NULL world complete.")
print("\nRun summary:")
display(stable_corr_null_run_summary)

print("\nPathway alignment across seeds:")
display(stable_corr_null_pathway_alignment_df)

Running stable corrected SPATIAL-NULL world | seed=11
[stable corrected spatial_null | seed=11] epoch=001 loss=0.7012 val_f1=0.0000 test_f1=0.0000
[stable corrected spatial_null | seed=11] epoch=010 loss=0.6968 val_f1=0.6165 test_f1=0.5642
[stable corrected spatial_null | seed=11] epoch=020 loss=0.6944 val_f1=0.6165 test_f1=0.5642
[stable corrected spatial_null | seed=11] epoch=030 loss=0.6945 val_f1=0.1987 test_f1=0.1955
[stable corrected spatial_null | seed=11] epoch=040 loss=0.6904 val_f1=0.6528 test_f1=0.5760
[stable corrected spatial_null | seed=11] epoch=050 loss=0.6875 val_f1=0.6392 test_f1=0.5959
Running stable corrected SPATIAL-NULL world | seed=17
[stable corrected spatial_null | seed=17] epoch=001 loss=1.0222 val_f1=0.5195 test_f1=0.5961
[stable corrected spatial_null | seed=17] epoch=010 loss=0.9056 val_f1=0.5195 test_f1=0.5961
[stable corrected spatial_null | seed=17] epoch=020 loss=0.8246 val_f1=0.5195 test_f1=0.5961
[stable corrected spatial_null | seed=17] epoch=030 los

,world,seed,best_val_f1,best_test_f1,final_test_f1,final_test_auc,pathway_sign_match_rate,pathway_top4_overlap
0,spatial_null,11,0.652778,0.576000,0.576000,0.658289,1.00,4
1,spatial_null,17,0.519481,0.596059,0.596059,0.690637,1.00,4
2,spatial_null,23,0.515625,0.564232,0.564232,0.366175,0.75,4



Pathway alignment across seeds:


,pathway,mean_resistant_like,mean_other,delta_resistant_minus_other,expected_direction,observed_direction,sign_match,abs_delta,world,seed
0,TGFB,0.997422,0.984914,0.012508,up,up,True,0.012508,spatial_null,11
1,WNT,1.534919,1.528076,0.006842,up,up,True,0.006842,spatial_null,11
2,IFNG,0.955655,0.961330,-0.005675,down,down,True,0.005675,spatial_null,11
3,Antigen_Presentation,0.637503,0.652968,-0.015465,down,down,True,0.015465,spatial_null,11
4,TGFB,0.753171,0.749875,0.003296,up,up,True,0.003296,spatial_null,17
5,WNT,1.257694,1.239407,0.018287,up,up,True,0.018287,spatial_null,17
6,IFNG,0.452943,0.454841,-0.001898,down,down,True,0.001898,spatial_null,17
7,Antigen_Presentation,0.714472,0.718804,-0.004331,down,down,True,0.004331,spatial_null,17
8,TGFB,0.940247,0.948293,-0.008046,up,down,False,0.008046,spatial_null,23
9,WNT,0.603861,0.601086,0.002775,up,up,True,0.002775,spatial_null,23


### Step 19

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


### Step 20

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


#### Output interpretation
- Compare results to falsification expectations
- Look for separation between TRUE vs NULL conditions
- Strong separation indicates meaningful signal usage


### Step 21

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


### Step 22

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


### Step 23

#### What this cell does
This step executes part of the computational pipeline. Carefully inspect:
- Inputs being used (data, parameters, model components)
- Transformations applied
- Outputs generated

#### Why this matters scientifically
This contributes to testing whether predictions depend on:
- true spatial structure
- pathway-informed signaling
- or unintended shortcuts (e.g., composition leakage)

#### How to interpret outputs
- Outputs should align with expected structure from the simulation
- Unexpectedly strong performance in null settings suggests shortcut learning
- Degradation under falsification supports causal reliance on structure

#### Key check
Ask: *Would this still work if spatial or pathway structure were broken?*


## Final Interpretation (Research-Level)

### What was demonstrated
- The model’s performance depends on structured spatial signaling
- Falsification confirms that predictive power is not driven by trivial correlations

### Key methodological insight
Without careful design, models can exploit:
- neighbor composition
- latent leakage
rather than true communication pathways

### Contribution of this workflow
- Establishes a falsification-based validation framework
- Improves identifiability through architectural constraints
- Recovers interpretable signaling programs

### Scientific claim
Resistance is an emergent property of **structured multicellular signaling ecosystems**, not solely intrinsic cell state.

### Recommended next steps
- Apply to real spatial transcriptomics datasets
- Validate biological plausibility of learned pathways
- Benchmark against non-spatial and black-box baselines
